<div align="center">

**Türkçe:** Bu eğitim Türkçe yürütülmektedir. [Bu Jupyter Notebook'un Türkçesini Colab'da açın](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-10/NB10_klinik_karar_destegi.ipynb) · [Türkçe ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-10/ders.html)

# Week 10: Clinical decision support systems (CDSS)

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lab.html) · [PDF notes](Week10_Lecture_Notes.pdf) · [Open in Colab](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-10/NB10_clinical_decision_support.ipynb)

</div>

**How to use this Jupyter notebook.** The notebook follows the practice stops of the lecture page. Every stop has three parts, and each part also runs on its own: Python basics, application with ready prompts, and the ready application. **Python basics** teaches the Python that the stop needs, with small examples from the topic of the week; its exercises check themselves. **Prompt and application** gives ready prompts for an artificial intelligence (AI) assistant: The assistant writes the code, you run it, and a check cell tells you whether the result is right. **Ready application** is the complete working Python program of the stop, with notes that explain it.

Go through the three parts in this order. If you already know Python, you may start with the prompts; if you only want to see the result, open the ready application. The weekly project at the end stands on its own as well.

**Start cells.** Every part opens with a start cell. It prepares the course package and the packages of the week, and it repeats the data calls and the code of the earlier parts that the cells of the part use. The part therefore runs without errors even when you start there. If you have run the earlier parts in this session, you do not need to run the start cell; running it again does no harm. A few sections of the ready applications, which the lecture page opens directly, have a start cell of their own.

**Data.** The records of the MIT-BIH Atrial Fibrillation Database are not part of the course repository. The start cells download the annotations of its 25 records and the signals of three of them from PhysioNet, about 86 MB, in one to three minutes, and check every file against the list of fingerprints of PhysioNet; files that are already there are skipped. The patients of the telemetry ward come with the course package.

**Finding your place.** The table of contents lists every stop and part. In Colab it opens with the list icon on the left; in Jupyter, with View → Table of Contents. Every part ends with a link back to the lecture page. In Colab, your changes are kept only in a copy saved to Drive: Use File → Save a copy in Drive.

**Where it runs.** The notebook runs in Jupyter (Anaconda) on your own computer and in Google Colab. Every data set here is synthetic or comes from a public research database. Real patient data belong only on a computer of your institution, under its rules, and never in a cloud notebook or in an AI chat.

## 0. Setup

**First use of Jupyter Notebook.** A notebook is made of cells. This cell holds text; the grey cells hold Python code. Click into the code cell below and press Shift+Enter: The cell runs, its result appears below it, and the number in the brackets on its left, such as `[1]`, shows the order of the run. The kernel, the Python process behind the notebook, remembers everything that the cells have run, so the cells run from top to bottom. **Kernel → Restart Kernel and Run All Cells** repeats the whole notebook, and Ctrl+S (Cmd+S on a Mac) saves it as an `.ipynb` file.

**The start cell.** The code cell below prepares the session. It finds the course package `vclinic` in the course folder; if the notebook was opened on its own, for example in Google Colab, it downloads the package from GitHub once. It then loads the packages, chooses the language of the messages and prepares the expected answers of the exercises. Its last lines import the libraries of the week. Every part of the notebook opens with a start cell that holds the same setup, together with the data calls of the part.

In [ ]:
# Start cell of the Jupyter notebook
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# the libraries of the week

import hashlib
import json
import re

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

## Stop 1: The AF model of the prototype

*Lecture page: [Practice stop 1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#stop-1)*

This stop starts the mini project of the session, an AF detection CDSS, with the model that predicts. The minutes of the 25 records of the AFDB are described by four numbers of their RR intervals, a logistic regression learns AF from the training records, its threshold is chosen on the validation records, and the test records give the AF burden and the alarm episodes of each record.

**At the end of this stop:** The median CV of the AF minutes of the training records (0.222), the threshold of the model (0.37) and the false alarm episodes of the test records (22).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 1 · Python basics

This week the model leaves the notebook: It is kept in a file, its file gets a fingerprint, and its functions move into a module that a web interface can use. This part shows how a dictionary becomes a text in JSON and back, how `pathlib` writes and reads a file, how `hashlib` computes the fingerprint of a file, how the magic command `%%writefile` turns a cell into a file, and why version numbers are compared as tuples of numbers and not as texts.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 1 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# repeated from the earlier cells: the data and the code that the cells below use
import hashlib
import json

In [ ]:
# Python practice: a small example for learning the language
toy = {"name": "AF-RR", "version": "1.0.0", "features": ["hr", "cv"], "coef": {"hr": 2.5, "cv": 3.3}, "intercept": -3.4, "threshold": 0.37}
text = json.dumps(toy, indent=2, sort_keys=True)
print(text)
back = json.loads(text)
print(type(text).__name__, type(back).__name__, back == toy, back["coef"]["cv"])

**Python note.** A dictionary can hold other dictionaries and lists: `toy["coef"]` is itself a dictionary, and `toy["coef"]["cv"]` reaches one of its values. `json.dumps` turns the dictionary into a text in JSON, a format that any program can read; `indent=2` puts every entry on its own line, and `sort_keys=True` writes the keys in alphabetical order, so that the same dictionary always gives the same text. `json.loads` turns the text back into a dictionary, and `back == toy` checks that nothing was lost. `type(text).__name__` gives the name of the type of a value: `str` for a text, `dict` for a dictionary.

In [ ]:
# Python practice: a file and its fingerprint
path = pathlib.Path("toy_model.json")
path.write_text(text, encoding="utf-8")
print(path.name, path.stat().st_size, "bytes")
digest = hashlib.sha256(path.read_bytes()).hexdigest()
print("SHA-256", digest)
changed = dict(toy, threshold=0.38)
digest2 = hashlib.sha256(json.dumps(changed, indent=2, sort_keys=True).encode("utf-8")).hexdigest()
print("SHA-256", digest2)
print(sum(a == b for a, b in zip(digest, digest2)), "of 64 digits agree in place")

**Python note.** `pathlib.Path("toy_model.json")` names a file in the folder of the notebook. `write_text` writes a text into it, `read_bytes` reads its bytes back, and `stat().st_size` gives its size in bytes. `hashlib.sha256(...)` computes the fingerprint of bytes, and `hexdigest()` writes it as 64 hexadecimal digits, the digits 0 to 9 and the letters a to f. A text becomes bytes with `.encode("utf-8")`. `dict(toy, threshold=0.38)` makes a copy of the dictionary with one value changed, and `zip(digest, digest2)` pairs the digits of the two fingerprints one by one.

In [ ]:
%%writefile toy_tools.py
# Python practice: a small module of your own
"""Tools of the toy model."""


def version_tuple(version):
    """'1.10.0' becomes (1, 10, 0), so that versions can be compared as numbers."""
    return tuple(int(part) for part in version.split("."))


def alarm(probability, threshold=0.37):
    """True when the probability reaches the threshold."""
    return probability >= threshold

In [ ]:
# Python practice: using the module
import importlib
import toy_tools
importlib.reload(toy_tools)
print(toy_tools.version_tuple("1.10.0"), toy_tools.alarm(0.40), toy_tools.alarm(0.40, threshold=0.5))
print("as texts:", "1.10.0" > "1.9.0", "| as tuples:", toy_tools.version_tuple("1.10.0") > toy_tools.version_tuple("1.9.0"))

**Python note.** A cell that starts with `%%writefile toy_tools.py` is not run: Jupyter writes its text into the file `toy_tools.py`, which then holds a small *module*. `import toy_tools` reads the module, and `toy_tools.alarm(...)` calls one of its functions. `importlib.reload` reads the file again after it changed; without it, Python keeps the first version it read. The function `alarm` has a *default value* for `threshold`: When the call does not give it, it is 0.37. The text in triple quotes under a `def` line is the *docstring* of the function, which `help(toy_tools.alarm)` shows. Texts are compared letter by letter, so the text "1.10.0" comes before "1.9.0"; tuples of numbers are compared number by number, and (1, 10, 0) comes after (1, 9, 0).

**Quick check.** Why does the model file of this week use `sort_keys=True` when it is written?

<details><summary>Answer</summary>

So that the same model always gives the same text, and therefore the same fingerprint. Without it, two programs that put the keys in another order would write two different files, with two fingerprints, for one model.

</details>

**Exercise 1.1.** Write `toy` as JSON without `indent` and with sorted keys. How many characters does the text have?

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(json.dumps(toy, sort_keys=True))
teach.check("Exercise 1.1", answer)

**Exercise 1.2.** Make a copy of `toy` with the intercept −3.41, write both as JSON with `indent=2` and sorted keys, and compute their SHA-256 fingerprints. In how many of the 64 places do the two fingerprints have the same digit?

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
a = hashlib.sha256(json.dumps(toy, indent=2, sort_keys=True).encode("utf-8")).hexdigest()
b = hashlib.sha256(json.dumps(dict(toy, intercept=-3.41), indent=2, sort_keys=True).encode("utf-8")).hexdigest()
answer = sum(x == y for x, y in zip(a, b))
teach.check("Exercise 1.2", answer)

**Exercise 1.3.** Sort the versions "1.2.0", "1.10.0", "1.9.3" and "1.10.1" from the oldest to the newest with `toy_tools.version_tuple`. At which position, counting from 1, does "1.9.3" stand?

In [ ]:
answer = None   # your answer
teach.check("Exercise 1.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
order = sorted(["1.2.0", "1.10.0", "1.9.3", "1.10.1"], key=toy_tools.version_tuple)
answer = order.index("1.9.3") + 1
teach.check("Exercise 1.3", answer)

**«Stop 1 · Python basics» is complete.** Next comes «Stop 1 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#from-notebook-to-application-streamlit) and continue reading at «Developing a web interface with Streamlit».

### Stop 1 · Prompt and application

This part has three prompts: «Four numbers of a minute», «The model and its threshold» and «AF burden and alarm episodes of the test records».

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Working with prompts.** An AI assistant that can write Python code is enough. Gemini is built into Google Colab and opens with the Gemini button at the top of the window; ChatGPT, Claude, Copilot or Gemini in another browser tab work as well. All prompts of the week are also in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-10/PROMPTS.md), where GitHub shows a copy button next to each one.

**Two names in every prompt: "data" and "answer".** An AI assistant usually does not see the notebook, so every prompt describes the data in words. For each prompt, the data cell puts its data into a table named "data", and the prompt describes the rows and the columns of this table. The prompt also asks the assistant to keep the result under the name "answer", and the check cell reads the result under this name. These are the only two names from the code that appear in the prompts of the course.

**Repair prompt** (for code that runs but does not pass the check):

> The code below was written for this task: [paste the Task line of the prompt]. It runs, but the check of the course prints "not yet". Find the error, explain it in two sentences and return the corrected code. The result must still be kept under the name "answer".  
>
> [paste the code here]  

**Prompt for an error message** (for code that stops with an error):

> Running the code below in a Jupyter notebook gives the error message that follows. Explain the cause in plain language and return the corrected code.  
>
> [paste the code here]  
>
> [paste the error message here]  

**A rule that never changes.** The prompts contain only the synthetic or public data of the course. Never paste real patient data into an AI assistant.

**Start cell.** For «Stop 1 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

from vclinic import afdb, holter
import hashlib
import json
from sklearn.linear_model import LogisticRegression

_SETS = {"training": "training", "validation": "validation", "test": "test"}


def _course():
    """The minutes of the 25 records of the AFDB and of the four patients of the telemetry ward, with their RR intervals, their five numbers and the probabilities of version 1.0 (computed once, about twenty seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "AF from RR intervals", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """The probability of AF for every minute of a table."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Runs of consecutive alarm minutes, as (first, last) positions."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """A table of the minutes of the AFDB with the chosen columns."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"record": m["record"], "set": m["set"].map(_SETS), "start (s)": m["start_s"].round(2),
            "AF": m["af"], "AF share": m["af_share"].round(3), "RR intervals (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "heart rate": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "probability": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)


_c = _course()
print(len(_c["minutes"]), "minutes of the 25 records of the AFDB")

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.metrics import roc_auc_score, roc_curve

#### Prompt 1.1: Four numbers of a minute

**Goal.** Describe every minute of the AFDB by four numbers of its RR intervals.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist of a telemetry ward wants a decision support prototype that flags minutes of possible AF from the times of the beats. The first step is to describe each minute by numbers that measure how irregular its RR intervals are.  
> Data: The notebook holds a table named "data" with the minutes of the MIT-BIH Atrial Fibrillation Database (AFDB), 25 Holter recordings of about ten hours of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. Each row is a minute: "record" gives the recording, "set" its set ("training", 15 records; "validation", 5 records; "test", 5 records), and "start (s)" the start of the minute in seconds from the start of the recording. "AF" is 1 when more than half of the minute lies in AF as marked by the experts, and 0 otherwise. Each cell of "RR intervals (s)" holds the RR intervals of the minute in seconds, in their order, as a list of 20 or more numbers.  
> Task: For every minute, compute four numbers from its RR intervals: the heart rate, 60 divided by the mean interval; the coefficient of variation, the standard deviation of the intervals (with n − 1) divided by their mean; the RMSSD divided by the mean interval, where the RMSSD is the square root of the mean of the squared differences between successive intervals; and the pNN50, the share of successive differences larger than 0.05 seconds. Then compare the four numbers of the minutes with and without AF in the training records.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the median of each number for the training minutes with AF and without AF, and keep the median CV of the training minutes with AF, with three decimals, under the name "answer".  
> Test: The heart rate of every minute must lie between 20 and 250 per minute, and the pNN50 between 0 and 1.  

In [ ]:
# Data of Prompt 1.1: the table "data" that the prompt describes
data = _minutes_table(["record", "set", "start (s)", "AF", "RR intervals (s)"])
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.1
teach.check_step("Prompt 1.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.1 (run it only if the code of the assistant does not pass the check)
data = _minutes_table(["record", "set", "start (s)", "AF", "RR intervals (s)"])
d = data


def four(rr):
    rr = np.asarray(rr, dtype=float)
    diff = np.diff(rr)
    return pd.Series({"heart rate": 60 / rr.mean(), "CV": rr.std(ddof=1) / rr.mean(),
                      "RMSSD/RR": np.sqrt(np.mean(diff ** 2)) / rr.mean(), "pNN50": np.mean(np.abs(diff) > 0.05)})


numbers = d["RR intervals (s)"].apply(four)
table = pd.concat([d[["record", "set", "AF"]], numbers], axis=1)
train = table[table["set"] == "training"]
print(train.groupby("AF")[["heart rate", "CV", "RMSSD/RR", "pNN50"]].median().round(3))
answer = round(float(train.loc[train["AF"] == 1, "CV"].median()), 3)
teach.check_step("Prompt 1.1", "answer", globals())

**What the result shows.** The median CV of the AF minutes is 0.222, several times that of the other minutes: The RR intervals of AF vary a lot from beat to beat, and the four numbers capture it.

#### Prompt 1.2: The model and its threshold

**Goal.** Fit version 1.0 of the AF model on the training records and choose its threshold on the validation records.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants the prototype to miss as few AF minutes as possible: At least 95 percent of them should raise an alarm. The false alarms are the price, and the test records must stay untouched for the final judgement.  
> Data: The notebook holds a table named "data" with the minutes of the MIT-BIH Atrial Fibrillation Database (AFDB), 25 Holter recordings of about ten hours of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. Each row is a minute: "record" gives the recording, "set" its set ("training", 15 records; "validation", 5 records; "test", 5 records) and "start (s)" the start of the minute in seconds. "AF" is 1 when more than half of the minute lies in AF as marked by the experts, and 0 otherwise. Four numbers describe the RR intervals of the minute: "heart rate" (per minute), "CV" (the standard deviation of the intervals divided by their mean), "RMSSD/RR" (the root mean square of the successive differences divided by the mean interval) and "pNN50" (the share of successive differences larger than 50 ms).  
> Task: Fit a logistic regression for AF on the training minutes only, with at most 3,000 iterations. Its inputs are the heart rate, the natural logarithm of the CV, the natural logarithm of the RMSSD/RR and the pNN50, each turned into a z-score with the mean and the standard deviation of the training minutes. Compute the probability of AF for every validation minute and the area under the ROC curve (AUC) on the validation minutes. Choose the threshold on the validation minutes only: Among 0.05, 0.06, 0.07 and so on up to 0.95, take the highest value at which at least 95 percent of the validation minutes with AF have a probability at or above it.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the validation AUC, the threshold and the coefficients of the model, and keep the threshold, with two decimals, under the name "answer".  
> Test: No test minute may be used, and the share of the validation minutes with AF at or above the threshold must be at least 0.95.  

In [ ]:
# Data of Prompt 1.2: the table "data" that the prompt describes
data = _minutes_table(["record", "set", "start (s)", "AF", "heart rate", "CV", "RMSSD/RR", "pNN50"])
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.2
teach.check_step("Prompt 1.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.2 (run it only if the code of the assistant does not pass the check)
data = _minutes_table(["record", "set", "start (s)", "AF", "heart rate", "CV", "RMSSD/RR", "pNN50"])
d = data
X = pd.DataFrame({"heart rate": d["heart rate"], "log CV": np.log(d["CV"]),
                    "log RMSSD/RR": np.log(d["RMSSD/RR"]), "pNN50": d["pNN50"]})
tr_ = (d["set"] == "training").to_numpy()
va_ = (d["set"] == "validation").to_numpy()
mean, sd = X[tr_].mean(), X[tr_].std()
lr = LogisticRegression(max_iter=3000).fit((X[tr_] - mean) / sd, d["AF"][tr_])
p = lr.predict_proba((X[va_] - mean) / sd)[:, 1]
y = d["AF"].to_numpy()[va_]
grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
threshold = max(t for t in grid if (p[y == 1] >= t).mean() >= 0.95)
print("validation AUC", round(roc_auc_score(y, p), 3), "| threshold", threshold)
print("coefficients", dict(zip(X.columns, np.round(lr.coef_[0], 3))), "| intercept", round(lr.intercept_[0], 3))
answer = round(float(threshold), 2)
teach.check_step("Prompt 1.2", "answer", globals())

**What the result shows.** The threshold is 0.37. It keeps a sensitivity of at least 95 percent on the validation records; how many false alarms it costs, the next step and the user test show.

#### Prompt 1.3: AF burden and alarm episodes of the test records

**Goal.** Turn the probabilities of the test minutes into alarm episodes and an AF burden for every test record.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist does not review single minutes but runs of alarm minutes, and wants the AF burden of each recording as the report will give it.  
> Data: The notebook holds a table named "data" with the minutes of the 5 test records of the MIT-BIH Atrial Fibrillation Database, Holter recordings of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. Each row is a minute: "record" gives the recording and "start (s)" the start of the minute in seconds. "AF" is 1 when more than half of the minute lies in AF as marked by the experts, "AF share" is the share of the minute in AF, from 0 to 1, and "probability" is the probability of AF that a model of the course gives the minute. The model raises an alarm for a minute when its probability is at least 0.37.  
> Task: Within each record, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm, and a false alarm episode is an episode in which no minute has AF. For each record, compute the AF burden of the annotation, 100 times the mean AF share of its minutes; the AF burden of the model, 100 times the share of its minutes with an alarm; the number of alarm episodes; and the number of false alarm episodes.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each record, and keep the total number of false alarm episodes of the test records under the name "answer".  
> Test: The alarm episodes of a record cannot be more than half of its minutes plus one, and the burdens must lie between 0 and 100.  

In [ ]:
# Data of Prompt 1.3: the table "data" that the prompt describes
data = _minutes_table(["record", "start (s)", "AF", "AF share", "probability"], sets=("test",))
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 1.3
teach.check_step("Prompt 1.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 1.3 (run it only if the code of the assistant does not pass the check)
data = _minutes_table(["record", "start (s)", "AF", "AF share", "probability"], sets=("test",))


def episodes(alarm, af):
    runs, first = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    return len(runs), sum(1 for s, e in runs if af[s:e + 1].sum() == 0)


d = data.sort_values(["record", "start (s)"])
threshold = 0.37
rows = []
for record, part in d.groupby("record"):
    alarm = part["probability"].to_numpy() >= threshold
    n, false = episodes(alarm, part["AF"].to_numpy())
    rows.append({"record": record, "minutes": len(part),
                         "AF burden, annotation (%)": round(100 * part["AF share"].mean(), 1),
                         "AF burden, model (%)": round(100 * alarm.mean(), 1),
                         "alarm episodes": n, "false alarm episodes": false})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
answer = int(table["false alarm episodes"].sum())
teach.check_step("Prompt 1.3", "answer", globals())

**What the result shows.** The test records give 22 false alarm episodes in about 50 hours. The burdens of the model follow those of the annotation closely in most records, as the report of the prototype needs.

**«Stop 1 · Prompt and application» is complete.** Next comes «Stop 1 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#from-notebook-to-application-streamlit) and continue reading at «Developing a web interface with Streamlit».

### Stop 1 · Ready application

This part reads the records of the MIT-BIH Atrial Fibrillation Database with their rhythms, cuts their RR intervals into windows of one minute, computes five numbers for every minute, and builds the AF model of the prototype with its threshold, its alarm episodes and the AF burden.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 1 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 1 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

#### The records and their rhythms

*Lecture page: [AF from RR intervals: Five numbers of a minute](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#af-from-rr-intervals-five-numbers-of-a-minute)*

The MIT-BIH Atrial Fibrillation Database holds 25 Holter recordings of about ten hours. Experts marked the rhythm by hand with four labels: `(AFIB` for atrial fibrillation, `(AFL` for atrial flutter, `(J` for AV junctional rhythm and `(N` for all other rhythms. The opening parenthesis is part of each label in the annotation files. The beats were found by an automatic detector, as in a recorder of a ward. The course package reads the annotations with `afdb.rhythm`, which gives the episodes of each record as (start, end, rhythm) in seconds, and `afdb.SPLITS` holds the split of the course by record.

**From the lecture to the code.** The split by record: `SPLIT`. The rhythm episodes of every record: `episodes`. The hours of each rhythm per record and the set of each record: `hours`.

In [ ]:
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                        for record in afdb.RECORDS for start, end, name in episodes[record]])
hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
hours["set"] = hours.index.map(SPLIT)

print(hours.groupby("set")[["(AFIB", "(AFL", "(J", "(N"]].sum().round(1))

**Python note.** `afdb.SPLITS` is a dictionary from the name of a set to its records, and the *dictionary comprehension* in the first line turns it around: `SPLIT["08219"]` gives the set of record 08219. `episodes` holds the list of episodes of every record. The list comprehension inside `pd.DataFrame([...])` makes one row per episode, with its length in hours. `pivot_table` then adds the hours of each rhythm for every record: one row per record, one column per rhythm, and `fill_value=0.0` where a record has no episode of a rhythm. `groupby("set")` adds the rows of each set.

In [ ]:
order = hours.sort_values(["set", "(AFIB"]).index
fig, ax = plt.subplots(figsize=(9.6, 3.2))
bottom = np.zeros(len(order))
for rhythm, colour in (("(AFIB", "#9B1B1B"), ("(AFL", "#B97813"), ("(J", "#0E7A78"), ("(N", "#C8C0B8")):
    ax.bar(range(len(order)), hours.loc[order, rhythm], bottom=bottom, color=colour, label=rhythm)
    bottom += hours.loc[order, rhythm].to_numpy()
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{r}\n{hours.loc[r, 'set'][:3]}" for r in order], fontsize=7)
ax.set_ylabel("hours")
ax.legend(fontsize=8, ncol=4)
ax.set_title("Hours of each rhythm in the 25 records of the AFDB (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
plt.tight_layout()
plt.show()

**Observe and reflect.** The share of AF differs a lot between the records: some hold a few minutes, others almost the whole recording. Why does the course split the data by record and not by minute, and what would a split by minute do to the test AUC?

**Exercise A.** How many hours of AF, marked `(AFIB`, do the validation records hold? Keep the number with one decimal.

In [ ]:
data = exercise_table("A", hours=hours)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(hours.loc[hours["set"] == "validation", "(AFIB"].sum()), 1)
teach.check("Exercise A", answer)

#### RR intervals in windows of one minute

*Lecture page: [AF from RR intervals: Five numbers of a minute](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#af-from-rr-intervals-five-numbers-of-a-minute)*

The prototype judges every minute. `minutes_of` cuts the beats of a recording into windows of one minute and keeps the RR intervals that end in each window. A minute with fewer than 20 intervals, or with an interval longer than 3 seconds, is left out. When the rhythm is known, the function also gives the share of the minute in AF, and a minute counts as AF when this share is above one half.

**From the lecture to the code.** The intervals $RR_i$ of a minute: `x`. The share of the minute in AF: `af_share`. The label of the minute, 1 for AF: `af`.

In [ ]:
WINDOW_S = 60


def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows

minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)

print(len(minutes), "minutes")
print(minutes.groupby("set")["af"].agg(["count", "sum"]).rename(columns={"count": "minutes", "sum": "AF minutes"}))

**Python note.** `np.diff(beat_s)` gives the intervals between successive beats, and `beat_s[1:]` the time at which each interval ends. `np.arange(first, last, WINDOW_S)` gives the start of every window. Inside the loop, the *mask* `(rr_end >= start) & (rr_end < start + WINDOW_S)` keeps the intervals that end in the window. `continue` skips the rest of the loop for a minute that is left out. The share of AF adds, for each episode marked `(AFIB`, the part that overlaps the window: `max(0.0, min(e, start + 60) - max(s, start))`. `astype(int)` turns True and False into 1 and 0.

In [ ]:
record = "04126"
part = minutes[minutes["record"] == record]
pair = [part[part["af"] == 0].iloc[0], part[part["af"] == 1].iloc[0]]
fig, axes = plt.subplots(1, 2, figsize=(9.6, 2.8), sharey=True)
for ax, row, name in zip(axes, pair, ["no AF", "AF"]):
    ax.plot(row["rr"], marker="o", ms=3, color="#9B1B1B" if row["af"] else "#1F5F8B")
    ax.set_title(f"{record}, {row['start_s'] / 3600:.2f} h: {name}", fontsize=10)
    ax.set_xlabel("interval of the minute")
axes[0].set_ylabel("RR (s)")
plt.tight_layout()
plt.show()

**Observe and reflect.** In the minute without AF the intervals change slowly, and in the minute with AF every interval differs from the one before. What would a run of premature atrial beats look like in the same plot?

**Exercise B.** How many minutes of the test records count as AF?

In [ ]:
data = exercise_table("B", minutes=minutes)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(minutes.loc[minutes["set"] == "test", "af"].sum())
teach.check("Exercise B", answer)

#### Five numbers of a minute

*Lecture page: [AF from RR intervals: Five numbers of a minute](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#af-from-rr-intervals-five-numbers-of-a-minute)*

Four numbers measure how much the intervals of a minute vary: the heart rate, the coefficient of variation, the RMSSD divided by the mean interval and the pNN50. The fifth, the sample entropy, measures whether the variation repeats a pattern.

**From the formulas to the code.** $\mathrm{HR} = 60/\overline{RR}$: `60 / rr.mean()`. $\mathrm{CV} = s_{RR}/\overline{RR}$: `rr.std(ddof=1) / rr.mean()`. $\mathrm{RMSSD}/\overline{RR}$: `np.sqrt(np.mean(d ** 2)) / rr.mean()` with the successive differences `d`. pNN50: `np.mean(np.abs(d) > 0.05)`. In `sample_entropy`, the counts $B$ and $A$: `b` and `a`, with $m$ = `m` and $r$ = `r`.

**Start cell.** For the section «Five numbers of a minute», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «Five numbers of a minute»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)

**On the lecture page:** [Worked example: The sample entropy of three short minutes](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#worked-example-the-sample-entropy-of-three-short-minutes). The cell below computes the sample entropy of every minute with the same rule.

In [ ]:
def sample_entropy(rr, m=1, r=0.03):
    """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
    n = len(rr)

    def pairs(length):
        t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
        alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
        return (alike.sum() - len(t)) / 2

    b, a = pairs(m), pairs(m + 1)
    return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


def features(rr):
    """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
    d = np.diff(rr)
    return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

for name, rr in (("sinus rhythm", [0.80, 0.81, 0.82, 0.81, 0.80, 0.81, 0.82, 0.81, 0.80]),
                 ("premature atrial beats", [0.80, 0.81, 0.55, 1.05, 0.80, 0.81, 0.56, 1.04, 0.80]),
                 ("AF", [0.61, 0.53, 0.84, 0.62, 0.85, 0.60, 0.97, 0.83, 0.99])):
    f = features(np.array(rr))
    print(f"{name:28}", {k: round(v, 3) for k, v in f.items()})

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c134)*

**Python note.** `np.lib.stride_tricks.sliding_window_view(rr, length)` gives every stretch of `length` consecutive intervals as one row of a table, and `[: n - m]` keeps the first $n - m$ stretches, the same number for both lengths. `t[:, None, :] - t[None, :, :]` compares every stretch with every other at once: a table of differences of size stretches × stretches × length. `.max(axis=2) <= r` says for every pair whether all of its intervals are within $r$. The pairs of a stretch with itself lie on the diagonal; `alike.sum() - len(t)` removes them, and `/ 2` counts each pair once. The three short minutes are those of the worked example of the lecture.

**On the lecture page:** [Worked example: Seven intervals of sinus rhythm and of AF](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#worked-example-seven-intervals-of-sinus-rhythm-and-of-af). The cell below computes the same numbers for every minute of the records.

In [ ]:
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

train = minutes[minutes["set"] == "training"]
print(train.groupby("af")[["hr", "cv", "rmssd", "pnn50", "sampen"]].median().round(3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c136)*

In [ ]:
names = {"hr": "heart rate (per minute)", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50", "sampen": "sample entropy"}
fig, axes = plt.subplots(1, 5, figsize=(11, 2.8))
for ax, (key, label) in zip(axes, names.items()):
    ax.boxplot([train.loc[train["af"] == 0, key], train.loc[train["af"] == 1, key]], showfliers=False)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["no AF", "AF"])
    ax.set_title(label, fontsize=9)
    if key in ("cv", "rmssd"):
        ax.set_yscale("log")
plt.suptitle("The five numbers of the training minutes (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
plt.tight_layout()
plt.show()

**Observe and reflect.** The boxes of the CV, the RMSSD/RR and the pNN50 barely overlap, and those of the heart rate overlap a lot. Which minutes without AF would you expect in the upper whiskers of the three numbers of irregularity?

**Advanced (optional).** The tolerance $r$ of 0.03 seconds is fixed in seconds. Lake and Moorman also studied tolerances relative to the mean interval for very short series. Try `sample_entropy(x, r=0.02)` and `r=0.05` on the three short minutes and see which counts change.

**Exercise C.** What is the median sample entropy of the training minutes with AF? Keep it with two decimals.

In [ ]:
data = exercise_table("C", minutes=minutes)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(float(train.loc[train["af"] == 1, "sampen"].median()), 2)
teach.check("Exercise C", answer)

#### The model, its threshold and the AF burden

*Lecture page: [From probabilities to alarms and the AF burden](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#from-probabilities-to-alarms-and-the-af-burden)*

Version 1.0 of the model is a logistic regression on four numbers: the heart rate, the logarithms of the CV and of the RMSSD/RR, and the pNN50, each as a z-score of the training minutes. Its threshold is the highest one at which at least 95 percent of the AF minutes of the validation records raise an alarm. Runs of consecutive alarm minutes are the alarm episodes, and the share of minutes with an alarm is the AF burden.

**From the formulas to the code.** $g(x_j)$: `design`. $\mu_j$ and $\sigma_j$: `mean` and `sd` in `fit_model`. $z_j$ and $p = 1/(1 + e^{-\eta})$: `af_probability`. The threshold $t$: `choose_threshold`. The alarm episodes: `alarm_episodes`. The AF burden and the false alarm episodes of every record: `by_record`.

**Start cell.** For the section «The model, its threshold and the AF burden», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «The model, its threshold and the AF burden»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)
def sample_entropy(rr, m=1, r=0.03):
    """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
    n = len(rr)

    def pairs(length):
        t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
        alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
        return (alike.sum() - len(t)) / 2

    b, a = pairs(m), pairs(m + 1)
    return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))
def features(rr):
    """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
    d = np.diff(rr)
    return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

**On the lecture page:** [Worked example: Ten minutes, three episodes](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#worked-example-ten-minutes-three-episodes). The cell below turns the probabilities of the model into alarms, alarm episodes and the AF burden.

In [ ]:
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]


def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X


def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


def choose_threshold(p, y, sensitivity=0.95):
    """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


def alarm_episodes(alarm):
    """Runs of consecutive alarm minutes, as (first, last) positions."""
    runs, first = [], None
    for k, a in enumerate(alarm):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    if first is not None:
        runs.append((first, len(alarm) - 1))
    return runs


def by_record(table, p, threshold):
    """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
    rows = []
    for record, part in table.assign(p=p).groupby("record", sort=True):
        alarm = (part["p"] >= threshold).to_numpy()
        runs = alarm_episodes(alarm)
        false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
        rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                     "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
    return pd.DataFrame(rows).set_index("record")

training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())

print("coefficients", model["coef"], "| intercept", model["intercept"])
print("validation AUC", round(roc_auc_score(validation["af"], p_val), 3), "| threshold", model["threshold"])
records_val = by_record(validation, p_val, model["threshold"])
print(records_val.round(1))
print("alarm episodes", int(records_val["episodes"].sum()), "| false", int(records_val["false_episodes"].sum()))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c146)*

**Python note.** The model is a plain dictionary that holds its name, version, features, means, standard deviations, coefficients and intercept. The numbers are rounded to six decimals, which keeps the model file that «Stop 2 · Ready application» writes short and readable. `(X - mean) / sd` turns every column into z-scores at once, because pandas matches the columns by their names. In `af_probability`, `z[names].to_numpy() @ w` multiplies every row of z-scores with the coefficients and adds them up: `@` is the product of a matrix and a vector. In `choose_threshold`, `max(t for t in grid if ...)` takes the largest threshold of the grid that keeps the sensitivity. `groupby("record", sort=True)` goes through the records one by one, and `alarm_episodes` walks through the minutes of each record to find the runs.

In [ ]:
part = validation[validation["record"] == "08219"]
p = af_probability(model, part)
fig, ax = plt.subplots(figsize=(9.6, 2.8))
ax.fill_between(part["start_s"] / 3600, 0, part["af"], color="#C8C0B8", step="post", label="AF in the annotation")
ax.plot(part["start_s"] / 3600, p, color="#9B1B1B", lw=0.9, label="probability, version 1.0")
ax.axhline(model["threshold"], color="#1F5F8B", ls="--", lw=0.9, label="threshold")
ax.set_xlabel("hours from the start of record 08219")
ax.set_ylabel("probability of AF")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

**Observe and reflect.** Record 08219 gives many alarms outside the grey AF episodes. Before you look at any strip, what would you ask about these minutes?

**Exercise D.** Apply `by_record` to the validation records with the threshold 0.45 instead of the threshold of the model. How many false alarm episodes remain?

In [ ]:
data = exercise_table("D", p_val=p_val, validation=validation)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(by_record(validation, p_val, 0.45)["false_episodes"].sum())
teach.check("Exercise D", answer)

**Stop 1 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#from-notebook-to-application-streamlit) and continue reading at «Developing a web interface with Streamlit». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 2: The explanation, the report and the service

*Lecture page: [Practice stop 2](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#stop-2)*

This stop completes the mini project of the session and follows one alarm and one patient. A false alarm minute of record 08219 is explained by the SHAP values of its four numbers, the result of VC-0128 of the telemetry ward is written as an HL7 FHIR report, the beat times of the same patient are sent to the prediction service of the course, and one hour of the ECG of VC-0128 is uploaded, its beats found and its minutes judged by the service.

**At the end of this stop:** The probability of AF of the false alarm minute (0.985), the AF burden in the report of VC-0128 (15.5 percent), the AF minutes in the answer of the service (222) and the AF minutes of the uploaded hour of ECG (39).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 2 · Python basics

*Lecture page: [Why did it alarm? The SHAP explanation and the strip](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#why-did-it-alarm-the-explanation-and-the-strip)*

The ready application of this stop explains every alarm minute by what each number adds to its log-odds, writes the report as HL7 FHIR resources that refer to each other, and offers the model as a service that answers a request. The examples below introduce these constructs on small values: the contributions of a linear model, a nested resource with its references, a function from a request to an answer and a decorator that registers a function.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 2 · Python basics», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# repeated from the earlier cells: the data and the code that the cells below use
import json
import re

In [ ]:
# Python practice: a small example for learning the language
coef = {"hr": 0.8, "cv": 2.1, "rmssd": 1.4}   # the coefficients of the model
z = {"hr": 0.5, "cv": 1.8, "rmssd": 1.2}      # the z-scores of one minute
parts = {k: coef[k] * z[k] for k in coef}
log_odds = -3.4 + sum(parts.values())
print({k: round(v, 2) for k, v in parts.items()}, round(log_odds, 2), round(1 / (1 + np.exp(-log_odds)), 3))

**Python note.** For a linear model, what a number adds to the log-odds of a minute is its coefficient times its z-score, and the dictionary comprehension computes it for every number at once. The intercept plus the sum of the contributions is the log-odds of the minute, and the sigmoid turns it into the probability of AF, here 0.921. The function `contributions` of the ready application explains every alarm minute in this way; these contributions are the SHAP values of the model.

In [ ]:
# Python practice: a small example for learning the language
report = {"resourceType": "DiagnosticReport",
          "subject": {"reference": "Patient/VC-0128"},
          "result": [{"reference": "Observation/af-burden"}],
          "performer": [{"reference": "Device/af-rr"}]}
print(report["result"][0]["reference"])
print(re.findall(r'"reference": "([^"]+)"', json.dumps(report)))

**Python note.** A resource of HL7 FHIR is a dictionary that may hold other dictionaries and lists: `report["result"][0]["reference"]` goes from the list of results to its first element and from there to its reference. A reference names another resource by its type and its identifier, so the resources of a report point to each other instead of repeating each other. `json.dumps` writes the whole dictionary as text, and `re.findall` with a group collects every reference in it. The function `fhir_bundle` of the ready application builds the four resources of the report in this way.

In [ ]:
# Python practice: a small example for learning the language
def answer(request):
    """The answer of a small service: the request is a dictionary, and so is the answer."""
    beats = np.asarray(request["beat_s"], dtype=float)
    rr = np.diff(beats)
    return {"patient": request["patient"], "beats": int(len(beats)), "mean_rr_s": round(float(rr.mean()), 3)}


request = {"patient": "VC-0128", "beat_s": [0.0, 0.8, 1.7, 2.4]}
print(answer(request))
print(json.dumps(answer(request)))

**Python note.** A service takes a request and returns an answer, and both are dictionaries that can travel as JSON between two programs. The function `answer` reads the beat times of the request, computes the RR intervals with `np.diff` and returns what the caller needs; `int` and `float` turn the numbers of NumPy into plain numbers that `json.dumps` can write. The function `predict` of the ready application is the service of the prototype in this form, and the web server only passes its requests and answers on.

In [ ]:
# Python practice: a small example for learning the language
routes = {}


def post(path):
    def register(function):
        routes[path] = function
        return function
    return register


@post("/predict")
def predict_demo(request):
    return {"received": len(request["beat_s"])}


print(list(routes), routes["/predict"]({"beat_s": [0.0, 0.8, 1.7]}))

**Python note.** `@post("/predict")` above a function is a *decorator*: Python calls `post("/predict")`, which returns the function `register`, and passes the new function to it, so `register` keeps it in the dictionary `routes` under its address. A web framework works in the same way: In the ready application, `@api.post("/predict")` of Flask registers the function of the service as the answer to POST requests at /predict.

**Exercise 2.1.** A minute has the z-scores 1.2 for the heart rate, 2.0 for the CV and 0.5 for the RMSSD/RR. The coefficients are 0.6, 1.5 and 1.1 and the intercept −2.5. What is the probability of AF? Round to three decimals.

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
eta = -2.5 + 0.6 * 1.2 + 1.5 * 2.0 + 1.1 * 0.5
answer = round(float(1 / (1 + np.exp(-eta))), 3)
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** Add a second result, {"reference": "Observation/af-episodes"}, to a copy of the list `result` of the dictionary `report` above. How many references does the copy hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
new = dict(report, result=report["result"] + [{"reference": "Observation/af-episodes"}])
answer = len(re.findall(r'"reference": "([^"]+)"', json.dumps(new)))
teach.check("Exercise 2.2", answer)

**«Stop 2 · Python basics» is complete.** Next comes «Stop 2 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#last-two-hours-deployment-and-publishing) and continue reading at «Last two hours: Deployment and publishing».

### Stop 2 · Prompt and application

This part has four prompts: «Why the minute alarmed», «The report as HL7 FHIR resources», «A request to the prediction service» and «From an uploaded ECG to the AF minutes». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-10/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 2 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

from vclinic import afdb, holter
import hashlib
import json
from sklearn.linear_model import LogisticRegression

_SETS = {"training": "training", "validation": "validation", "test": "test"}


def _course():
    """The minutes of the 25 records of the AFDB and of the four patients of the telemetry ward, with their RR intervals, their five numbers and the probabilities of version 1.0 (computed once, about twenty seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "AF from RR intervals", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """The probability of AF for every minute of a table."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Runs of consecutive alarm minutes, as (first, last) positions."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """A table of the minutes of the AFDB with the chosen columns."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"record": m["record"], "set": m["set"].map(_SETS), "start (s)": m["start_s"].round(2),
            "AF": m["af"], "AF share": m["af_share"].round(3), "RR intervals (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "heart rate": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "probability": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)



def _start_service(port=8765):
    """Starts the prediction service of the course on this computer, at the given port, once per kernel."""
    if "_service" in globals():
        return _service
    import http.server
    import threading
    c = _course()

    class _Handler(http.server.BaseHTTPRequestHandler):
        def do_POST(self):
            body = self.rfile.read(int(self.headers["Content-Length"]))
            out = json.dumps(c["predict"](json.loads(body), c["model"], c["digest"])).encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(out)

        def log_message(self, *args):
            pass

    try:
        server = http.server.ThreadingHTTPServer(("127.0.0.1", port), _Handler)
    except OSError as error:
        raise RuntimeError("Port 8765 is in use, perhaps by another notebook. Close it, or restart this kernel, and run the cell again.") from error
    threading.Thread(target=server.serve_forever, daemon=True).start()
    globals()["_service"] = server
    return server


_c = _course()
print(len(_c["minutes"]), "minutes of the AFDB and", len(_c["ward_minutes"]), "minutes of the telemetry ward")

# repeated from the earlier cells: the data and the code that the cells below use
import re

#### Prompt 2.1: Why the minute alarmed

**Goal.** Compute what each of the four numbers adds to the log-odds of AF in one false alarm minute.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The physician opens a false alarm of record 08219 and asks which numbers of the minute made the model raise it, before looking at the strip.  
> Data: The notebook holds a table named "data" that describes one minute of record 08219 of the MIT-BIH Atrial Fibrillation Database, a minute without AF in which a logistic regression of the course raises an alarm. The first four rows are the four numbers of the minute: "number" names it, "value in the minute" gives its value, "logarithm first" says whether the model takes the natural logarithm of the value before anything else ("yes" or "no"), "mean, training" and "standard deviation, training" are the mean and the standard deviation of the value, or of its logarithm, over the training minutes, and "coefficient" is its coefficient in the model. The last row, "intercept", holds only the intercept of the model in "coefficient".  
> Task: For each of the four numbers, take the natural logarithm of the value when "logarithm first" says yes, subtract the mean and divide by the standard deviation to get its z-score, and multiply the z-score by the coefficient: This is its contribution to the log-odds. Add the intercept and the four contributions to get the log-odds of the minute, and turn it into a probability with 1 / (1 + e to the power of minus the log-odds).  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with the z-score and the contribution of each number, print the log-odds and the probability, and keep the probability, with three decimals, under the name "answer".  
> Test: The intercept plus the four contributions must give the log-odds, and the probability must lie between 0 and 1.  

In [ ]:
# Data of Prompt 2.1: the table "data" that the prompt describes
_m = _course()["model"]
_x = _course()["minutes"]
_x = _x[(_x["record"] == "08219") & (_x["af"] == 0) & (_x["p10"] >= _m["threshold"])]
_x = _x.iloc[len(_x) // 2]
_names = {"hr": "heart rate", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50"}
data = pd.DataFrame({"number": [_names[k] for k in _m["features"]] + ["intercept"],
                     "value in the minute": [round(float(_x[k]), 4) for k in _m["features"]] + [np.nan],
                     "logarithm first": ["yes" if k in ("cv", "rmssd") else "no" for k in _m["features"]] + [""],
                     "mean, training": [_m["mean"][k] for k in _m["features"]] + [np.nan],
                     "standard deviation, training": [_m["sd"][k] for k in _m["features"]] + [np.nan],
                     "coefficient": [_m["coef"][k] for k in _m["features"]] + [_m["intercept"]]})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.1
teach.check_step("Prompt 2.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.1 (run it only if the code of the assistant does not pass the check)
_m = _course()["model"]
_x = _course()["minutes"]
_x = _x[(_x["record"] == "08219") & (_x["af"] == 0) & (_x["p10"] >= _m["threshold"])]
_x = _x.iloc[len(_x) // 2]
_names = {"hr": "heart rate", "cv": "CV", "rmssd": "RMSSD/RR", "pnn50": "pNN50"}
data = pd.DataFrame({"number": [_names[k] for k in _m["features"]] + ["intercept"],
                     "value in the minute": [round(float(_x[k]), 4) for k in _m["features"]] + [np.nan],
                     "logarithm first": ["yes" if k in ("cv", "rmssd") else "no" for k in _m["features"]] + [""],
                     "mean, training": [_m["mean"][k] for k in _m["features"]] + [np.nan],
                     "standard deviation, training": [_m["sd"][k] for k in _m["features"]] + [np.nan],
                     "coefficient": [_m["coef"][k] for k in _m["features"]] + [_m["intercept"]]})
d = data
rows = d.iloc[:4]
x = np.where(rows["logarithm first"] == "yes", np.log(rows["value in the minute"]),
               rows["value in the minute"])
z = (x - rows["mean, training"]) / rows["standard deviation, training"]
contribution = rows["coefficient"] * z
eta = float(d.iloc[4]["coefficient"] + contribution.sum())
print(pd.DataFrame({"number": rows["number"], "z": z.round(3), "contribution": contribution.round(3)}).to_string(index=False))
print("log-odds", round(eta, 3), "| probability", round(1 / (1 + np.exp(-eta)), 3))
answer = round(float(1 / (1 + np.exp(-eta))), 3)
teach.check_step("Prompt 2.1", "answer", globals())

**What the result shows.** The minute gets a probability of 0.985, mostly through its pNN50: Its successive intervals differ a lot. The contributions say why the model alarmed, and the strip shows why the intervals differ, premature beats and not AF.

#### Prompt 2.2: The report as HL7 FHIR resources

**Goal.** Write the result of VC-0128 as an HL7 FHIR R4 report that the information system of a hospital can read.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The 24-hour report of the prototype must reach the record of the patient as a preliminary report, until a cardiologist confirms or rejects it, and it must say which model wrote it.  
> Data: The notebook holds a table named "data" with one row: the result of a decision support prototype of the course for VC-0128, a patient of the telemetry ward of the virtual clinic, a synthetic data set. "patient" gives the identifier of the patient, "AF burden (%)" the share of the analysed minutes with an alarm of atrial fibrillation (AF), "alarm episodes" the number of runs of consecutive alarm minutes and "longest episode (minutes)" the longest of them. "start of the recording" and "end of the recording" give the period of the 24-hour recording, "model", "version" and "SHA-256" name the model and the fingerprint of its file, and "time of the report" the time at which the report is written, all times in the ISO format with their offset from UTC.  
> Task: Write a FHIR Bundle of type "collection" with four resources, each with an id and a fullUrl under http://example.org/fhir. A Device "af-rr" with the name of the model, its version and a note with the SHA-256. An Observation "af-burden" with the status "preliminary", the category "procedure", a local code "af-burden" of the system http://example.org/vclinic with the text "AF burden", the patient as subject, the period of the recording, the AF burden as a quantity in the unit "%" of UCUM (system http://unitsofmeasure.org, code "%") and a reference to the Device. A DiagnosticReport "af-report" with the status "preliminary", the text "Holter recording, AF analysis", the patient, the time of the report, a reference to the Observation and a conclusion in words. A Provenance that names the report as its target, records the time of the report and gives the Device as its author. Then check that every reference inside the bundle points to a resource of the bundle, apart from the patient.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the Observation as JSON, print the number of resources and of references that do not resolve, and keep the value of the AF burden in the Observation under the name "answer".  
> Test: The bundle must hold four resources, no reference inside it may point to a missing resource, and both statuses must be "preliminary".  

In [ ]:
# Data of Prompt 2.2: the table "data" that the prompt describes
_c = _course()
_w = _c["ward_minutes"]
_p = _w[_w["record"] == "VC-0128"]["p10"].to_numpy() >= _c["model"]["threshold"]
_eps, _start = [], None
for _k, _a in enumerate(list(_p) + [False]):
    if _a and _start is None:
        _start = _k
    if not _a and _start is not None:
        _eps.append(_k - _start)
        _start = None
data = pd.DataFrame([{"patient": "VC-0128", "AF burden (%)": round(100 * float(_p.mean()), 1), "alarm episodes": len(_eps),
                      "longest episode (minutes)": max(_eps), "start of the recording": "2026-11-18T08:00:00+03:00",
                      "end of the recording": "2026-11-19T08:00:00+03:00", "model": _c["model"]["name"], "version": _c["model"]["version"],
                      "SHA-256": _c["digest"], "time of the report": "2026-11-19T09:30:00+03:00"}])
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.2
teach.check_step("Prompt 2.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.2 (run it only if the code of the assistant does not pass the check)
_c = _course()
_w = _c["ward_minutes"]
_p = _w[_w["record"] == "VC-0128"]["p10"].to_numpy() >= _c["model"]["threshold"]
_eps, _start = [], None
for _k, _a in enumerate(list(_p) + [False]):
    if _a and _start is None:
        _start = _k
    if not _a and _start is not None:
        _eps.append(_k - _start)
        _start = None
data = pd.DataFrame([{"patient": "VC-0128", "AF burden (%)": round(100 * float(_p.mean()), 1), "alarm episodes": len(_eps),
                      "longest episode (minutes)": max(_eps), "start of the recording": "2026-11-18T08:00:00+03:00",
                      "end of the recording": "2026-11-19T08:00:00+03:00", "model": _c["model"]["name"], "version": _c["model"]["version"],
                      "SHA-256": _c["digest"], "time of the report": "2026-11-19T09:30:00+03:00"}])
r = data.iloc[0]
status = "preliminary"
device = {"resourceType": "Device", "id": "af-rr", "deviceName": [{"name": r["model"], "type": "model-name"}],
                "version": [{"value": r["version"]}], "note": [{"text": "SHA-256 " + r["SHA-256"]}]}
observation = {"resourceType": "Observation", "id": "af-burden", "status": status,
                      "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
                      "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF burden"},
                      "subject": {"reference": "Patient/" + r["patient"]},
                      "effectivePeriod": {"start": r["start of the recording"], "end": r["end of the recording"]},
                      "valueQuantity": {"value": float(r["AF burden (%)"]), "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
                      "device": {"reference": "Device/af-rr"}}
report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status, "code": {"text": "Holter recording, AF analysis"},
                "subject": {"reference": "Patient/" + r["patient"]}, "issued": r["time of the report"],
                "result": [{"reference": "Observation/af-burden"}],
                "conclusion": f"AF burden {r['AF burden (%)']} %, {r['alarm episodes']} alarm episodes, the longest {r['longest episode (minutes)']} minutes."}
provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                     "recorded": r["time of the report"], "agent": [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]}
bundle = {"resourceType": "Bundle", "type": "collection",
                "entry": [{"fullUrl": f"http://example.org/fhir/{x['resourceType']}/{x['id']}", "resource": x}
                          for x in (device, observation, report, provenance)]}
present = {f"{e['resource']['resourceType']}/{e['resource']['id']}" for e in bundle["entry"]}
refs = re.findall(r'"reference": "([^"]+)"', json.dumps(bundle))
open_ = [x for x in refs if x.split("/")[0] not in ("Patient", "Practitioner") and x not in present]
print(json.dumps(observation, indent=2, ensure_ascii=False))
print(len(bundle["entry"]), "resources;", len(refs), "references, of which do not resolve:", len(open_))
answer = float(observation["valueQuantity"]["value"])
teach.check_step("Prompt 2.2", "answer", globals())

**What the result shows.** The report gives an AF burden of 15.5 percent, with the status preliminary. After the decision of the cardiologist, the status becomes final and the report names the cardiologist; Part B of the interactive lab shows each change.

#### Prompt 2.3: A request to the prediction service

**Goal.** Send the beat times of VC-0128 to the prediction service of the course and read its answer.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The web interface of the ward should not hold the model itself: It asks the prediction service, which answers with the result and the version of the model.  
> Data: The notebook holds a table named "data" with one column, "beat time (s)": the times of the 104,314 beats that the monitor found in the 24-hour recording of VC-0128, a patient of the telemetry ward of the virtual clinic, a synthetic data set, in seconds from the start of the recording. The data cell has also started a prediction service of the course on this computer at the address http://127.0.0.1:8765/predict. The service takes a request in JSON with the fields "patient_id" and "beat_times_s" (a list of beat times in seconds) and answers in JSON with the model, its version and fingerprint, the threshold, the minutes analysed, the minutes with an alarm of atrial fibrillation (AF), the AF burden in percent and the alarm episodes.  
> Task: Build the request for patient "VC-0128" with all beat times, send it to the service as JSON with the method POST, and read the answer as JSON. The service runs on this computer, so the request must not go through a proxy server.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the size of the request in kilobytes, the model, its version and the first twelve digits of its fingerprint, the minutes, the AF minutes and the AF burden, show the alarm episodes as a table, and keep the number of AF minutes under the name "answer".  
> Test: The AF burden in the answer must be 100 times the AF minutes divided by the minutes, rounded to one decimal.  

In [ ]:
# Data of Prompt 2.3: the table "data" that the prompt describes
_start_service()
data = pd.DataFrame({"beat time (s)": np.round(_course()["ward"]["VC-0128"]["detected_s"], 3)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.3
teach.check_step("Prompt 2.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.3 (run it only if the code of the assistant does not pass the check)
_start_service()
data = pd.DataFrame({"beat time (s)": np.round(_course()["ward"]["VC-0128"]["detected_s"], 3)})
request = {"patient_id": "VC-0128", "beat_times_s": [float(t) for t in data["beat time (s)"]]}
body = json.dumps(request).encode("utf-8")
send = urllib.request.Request("http://127.0.0.1:8765/predict", data=body, headers={"Content-Type": "application/json"})
local = urllib.request.build_opener(urllib.request.ProxyHandler({}))
reply = json.loads(local.open(send, timeout=120).read().decode("utf-8"))
print(round(len(body) / 1024), "kB sent ·", reply["model"], reply["version"], reply["sha256"][:12])
print("minutes", reply["minutes"], "| AF minutes", reply["af_minutes"], "| AF burden", reply["burden_percent"], "%")
print(pd.DataFrame(reply["episodes"]))
answer = int(reply["af_minutes"])
teach.check_step("Prompt 2.3", "answer", globals())

**What the result shows.** The service finds 222 AF minutes and names its model and version in the answer. A program that calls it needs no copy of the model, and a new version changes one file on the side of the service.

#### Prompt 2.4: From an uploaded ECG to the AF minutes

**Goal.** Find the beats of an uploaded ECG and let the prediction service of the course judge every minute of it.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: In the mini project, the physician uploads one hour of the ECG of VC-0128 instead of the beat times of the monitor, so the prototype must find the beats itself before the model can predict.  
> Data: The notebook holds a table named "data" with one hour of the ECG of VC-0128, a patient of the telemetry ward of the virtual clinic, a synthetic data set: one column, "mV", with 900,000 values of lead II in millivolts, sampled at 250 Hz, from 13:00 to 14:00. The first value lies 5 hours, 18,000 seconds, after the start of the recording at 08:00. The data cell has also started the prediction service of the course on this computer at the address http://127.0.0.1:8765/predict. The service takes a request in JSON with the fields "patient_id" and "beat_times_s" (a list of beat times in seconds from the start of the recording) and answers in JSON with the model, its version and fingerprint, the threshold, the minutes analysed, the minutes with an alarm of atrial fibrillation (AF), the AF burden in percent and the alarm episodes, each with its start in seconds ("start_s") and its minutes.  
> Task: Find the beats of the ECG. Filter the signal with a band-pass filter from 5 to 15 Hz, applied forwards and backwards so that the peaks do not shift. Take as R peaks the local maxima of the filtered signal that are higher than a third of its 99th percentile and at least 0.25 seconds apart. Turn their positions into times in seconds from the start of the recording: Divide by 250 and add 18,000. Then send the beat times of patient "VC-0128" to the prediction service as JSON with the method POST, without a proxy server, and read the answer as JSON.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the number of beats found, the minutes, the AF minutes and the AF burden, show the alarm episodes as a table with their start as a time of day (the recording starts at 08:00), and keep the number of AF minutes under the name "answer".  
> Test: One hour of ECG must give between 4,000 and 9,000 beats, and the AF burden in the answer must be 100 times the AF minutes divided by the minutes, rounded to one decimal.  

In [ ]:
# Data of Prompt 2.4: the table "data" that the prompt describes
_start_service()
from vclinic import holter as _holter
_t, _x = _holter.strip(_course()["ward"]["VC-0128"], 5 * 3600.0, duration_s=3600, fs=250)
data = pd.DataFrame({"mV": np.round(_x, 4)})
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 2.4
teach.check_step("Prompt 2.4", "answer", globals())

In [ ]:
#@title Ready code of Prompt 2.4 (run it only if the code of the assistant does not pass the check)
_start_service()
from vclinic import holter as _holter
_t, _x = _holter.strip(_course()["ward"]["VC-0128"], 5 * 3600.0, duration_s=3600, fs=250)
data = pd.DataFrame({"mV": np.round(_x, 4)})
from scipy.signal import butter, filtfilt, find_peaks
b, a = butter(2, [5, 15], btype="band", fs=250)
filtered = filtfilt(b, a, data["mV"].to_numpy())
peaks, _ = find_peaks(filtered, height=np.percentile(filtered, 99) / 3, distance=int(0.25 * 250))
beat_s = 18000 + peaks / 250
request = {"patient_id": "VC-0128", "beat_times_s": [round(float(t), 3) for t in beat_s]}
send = urllib.request.Request("http://127.0.0.1:8765/predict", data=json.dumps(request).encode("utf-8"), headers={"Content-Type": "application/json"})
local = urllib.request.build_opener(urllib.request.ProxyHandler({}))
reply = json.loads(local.open(send, timeout=120).read().decode("utf-8"))
print(len(beat_s), "beats found")
print("minutes", reply["minutes"], "| AF minutes", reply["af_minutes"], "| AF burden", reply["burden_percent"], "%")
episodes = pd.DataFrame(reply["episodes"])
if len(episodes):
    episodes["start"] = [f"{8 + int(s // 3600):02d}:{int(s % 3600 // 60):02d}" for s in episodes["start_s"]]
print(episodes)
answer = int(reply["af_minutes"])
teach.check_step("Prompt 2.4", "answer", globals())

**What the result shows.** The service finds 39 AF minutes in the uploaded hour, in one episode that starts at 13:20, when the AF of the recording begins. The same file can be uploaded in the web interface, which runs this chain and adds the SHAP values of every alarm and the FHIR report.

**«Stop 2 · Prompt and application» is complete.** Next comes «Stop 2 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#last-two-hours-deployment-and-publishing) and continue reading at «Last two hours: Deployment and publishing».

### Stop 2 · Ready application

This part explains an alarm minute with its SHAP values and its strip, keeps the model in a file with its fingerprint and its card, writes the report as HL7 FHIR resources, and offers the model as a prediction service, written into the two files of the prototype and run from end to end on an uploaded ECG.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 2 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 2 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# functions of earlier parts, kept unchanged in the course package
from vclinic.week10 import features, choose_threshold, alarm_episodes

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import hashlib
import json
import re
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]
def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X
def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}
def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))
training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())

#### Why did it alarm? The SHAP explanation and the strip

*Lecture page: [Why did it alarm? The SHAP explanation and the strip](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#why-did-it-alarm-the-explanation-and-the-strip)*

For a logistic regression, the log-odds of a minute are the intercept plus one term per number, and each term is what that number adds: These terms are the SHAP values of the minute. This section explains the false alarm minute in the middle of the false alarm minutes of record 08219, counted in the order of time, and shows ten seconds of its ECG next to its RR intervals. It is the minute of the lecture.

**From the formulas to the code.** The contribution $c_j = \beta_j z_j$: `contributions(model, row)`. The log-odds $\eta = \beta_0 + \sum_j c_j$: `model["intercept"] + sum(c.values())`. The strip: `afdb.strip(record, start_s)`, both ECG signals in millivolts.

**On the lecture page:** [Worked example: The contributions of the false alarm minute of record 08219](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#worked-example-the-contributions-of-the-false-alarm-minute-of-record-08219). The cell below computes the same contributions.

In [ ]:
def contributions(model, row):
    """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
    x = design(pd.DataFrame([row]), model["features"]).iloc[0]
    z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
    return {name: model["coef"][name] * z[name] for name in model["features"]}

false_08219 = validation[(validation["record"] == "08219") & (validation["af"] == 0)]
false_alarms = false_08219[af_probability(model, false_08219) >= model["threshold"]]
row = false_alarms.iloc[len(false_alarms) // 2]
c = contributions(model, row)
eta = model["intercept"] + sum(c.values())
print(f"minute at {row['start_s'] / 3600:.2f} h", {k: round(v, 3) for k, v in row[["hr", "cv", "rmssd", "pnn50"]].items()})
print({k: round(v, 2) for k, v in c.items()}, "| log-odds", round(eta, 2), "| probability", round(1 / (1 + np.exp(-eta)), 3))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c157)*

**Python note.** `false_08219` keeps the minutes of record 08219 without AF, and `false_alarms` those among them whose probability reaches the threshold, in the order of time. `len(false_alarms) // 2` is the position in the middle, and `.iloc` takes the row at a position. `contributions` builds a table of one row with `pd.DataFrame([row])`, so that the same `design` as for the whole table takes the logarithms. The *dictionary comprehension* in the last line rounds every value of the dictionary `c` for printing.

**The same values from the coalitions of Week 9.** For a linear model, the SHAP values with the training minutes as the background have a closed form: the coefficient times the z-score. The next cell computes them as in Week 9, over all 16 coalitions of the four numbers, and compares them with the contributions.

In [ ]:
import itertools
import math

names = model["features"]
z_training = (design(training, names) - pd.Series(model["mean"])) / pd.Series(model["sd"])
z_minute = ((design(pd.DataFrame([row]), names) - pd.Series(model["mean"])) / pd.Series(model["sd"])).iloc[0]
beta = np.array([model["coef"][name] for name in names])

values = {}
for mask in itertools.product([0, 1], repeat=len(names)):
    z = z_training.copy()
    for member, name in zip(mask, names):
        if member:
            z[name] = z_minute[name]
    values[mask] = (model["intercept"] + z[names].to_numpy() @ beta).mean()   # value of the coalition: the mean log-odds over the background

phi = np.zeros(len(names))
for mask, value in values.items():
    size = sum(mask)
    for j in range(len(names)):
        if mask[j] == 0:
            with_j = mask[:j] + (1,) + mask[j + 1:]
            weight = math.factorial(size) * math.factorial(len(names) - size - 1) / math.factorial(len(names))
            phi[j] += weight * (values[with_j] - value)
print("SHAP values of the minute:", {name: round(float(v), 2) for name, v in zip(names, phi)})
print("The same as the contributions:", np.allclose(phi, [c[name] for name in names], atol=0.001))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c160)*

**Python note.** `itertools.product([0, 1], repeat=4)` gives the 16 coalitions as tuples of 0 and 1, as in Week 9. In a coalition, the numbers marked 1 keep the z-scores of the minute and the others take those of every training minute; the mean log-odds over the training minutes is the value of the coalition. `@` multiplies the table of z-scores by the coefficients, row by row. The weights built with `math.factorial` are those of the Shapley formula. The mean z-score of the training minutes is 0 up to the rounding of the stored means, so `np.allclose(..., atol=0.001)` finds the contributions again, to three decimals.

In [ ]:
t, ecg = afdb.strip("08219", row["start_s"] + 20.0)
beats = afdb.beats("08219")
fig, axes = plt.subplots(3, 1, figsize=(9.6, 6.2), gridspec_kw={"height_ratios": [2, 1.2, 1.2]})
axes[0].plot(t - t[0], ecg[:, 0], color="#231A1A", lw=0.7)
for b in beats[(beats >= t[0]) & (beats < t[-1])]:
    axes[0].axvline(b - t[0], color="#B97813", lw=0.5, alpha=0.6)
axes[0].set_title("Record 08219, ten seconds of the false alarm minute (MIT-BIH AFDB, PhysioNet, ODC-By 1.0)", fontsize=10)
axes[0].set_xlabel("s")
axes[0].set_ylabel("mV")
axes[1].plot(row["rr"], marker="o", ms=3, color="#9B1B1B")
axes[1].set_ylabel("RR (s)")
axes[1].set_xlabel("interval of the minute")
axes[2].barh(list(c), list(c.values()), color=["#9B1B1B" if v > 0 else "#1F5F8B" for v in c.values()])
axes[2].axvline(0, color="grey", lw=0.8)
axes[2].set_xlabel("contribution to the log-odds of AF")
plt.tight_layout()
plt.show()

**Python note.** `afdb.strip` reads the samples of ten seconds from the file of the signals of a record and gives the time of every sample and both ECG signals, so `ecg[:, 0]` is the first signal. The orange lines mark the beats that the automatic detector of the database found. `gridspec_kw={"height_ratios": [2, 1.2, 1.2]}` gives the three panels different heights. In the bar chart, a list comprehension chooses red for a positive contribution and blue for a negative one.

**Observe and reflect.** Look at the strip: Which beats come early, do they have P waves, and are the beats between them sinus beats? What would you write as the reason for rejecting this alarm in the decision log?

**Exercise E.** Take the alarm minute with the highest probability among the AF minutes of record 08219 and compute its contributions. What is the contribution of the pNN50? Keep it with two decimals.

In [ ]:
data = exercise_table("E", af_probability=af_probability, model=model, validation=validation)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
af_08219 = validation[(validation["record"] == "08219") & (validation["af"] == 1)]
best = af_08219.index[af_probability(model, af_08219).argmax()]
answer = round(float(contributions(model, af_08219.loc[best])["pnn50"]), 2)
teach.check("Exercise E", answer)

#### The model file, its fingerprint and its card

*Lecture page: [The model file, its fingerprint and its card](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#versions-fingerprints-and-the-model-card)*

The model leaves the notebook as a file in JSON, and the SHA-256 of the file names it. `save_model` writes the file and returns its fingerprint; `load_model` reads it back and computes the fingerprint again, so that the prototype can show it. The model card describes the model in nine sections.

**From the lecture to the code.** The file of version 1.0: `af_model_v1.0.0.json`. Its fingerprint: `digest`. The model card: `card`.

**Start cell.** For the section «The model file, its fingerprint and its card», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «The model file, its fingerprint and its card»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# functions of earlier parts, kept unchanged in the course package
from vclinic.week10 import features, choose_threshold

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import hashlib
import json
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]
def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X
def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}
def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))
training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())

**On the lecture page:** [Worked example: One millionth changes the fingerprint](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#worked-example-one-millionth-changes-the-fingerprint). The cell below writes the model file and computes its fingerprint.

In [ ]:
def save_model(model, path):
    """Write the model as JSON and return the SHA-256 of the file, its fingerprint."""
    text = json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False)
    pathlib.Path(path).write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_model(path):
    """Read a model written by save_model; returns the model and the SHA-256 of the file."""
    text = pathlib.Path(path).read_text(encoding="utf-8")
    return json.loads(text), hashlib.sha256(text.encode("utf-8")).hexdigest()

digest = save_model(model, "af_model_v1.0.0.json")
loaded, digest_again = load_model("af_model_v1.0.0.json")
print(pathlib.Path("af_model_v1.0.0.json").stat().st_size, "bytes · SHA-256", digest)
print("same model:", loaded == model, "| same fingerprint:", digest == digest_again)

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c171)*

**Python note.** `save_model` writes the dictionary with `indent=2`, sorted keys and `ensure_ascii=False`, which keeps letters such as ı and ş as they are, and it computes the fingerprint of exactly the text that it wrote. `load_model` returns two values, the model and the fingerprint, and the line `loaded, digest_again = ...` unpacks them. `loaded == model` compares two dictionaries entry by entry, including the nested ones.

In [ ]:
v, t = validation, model["threshold"]
card = {
    "Model details": f"{model['name']}, version {model['version']}, logistic regression on {', '.join(model['features'])}; SHA-256 {digest[:12]}…",
    "Intended use": "Flags minutes of possible AF in the RR intervals of adults on telemetry, for review by a physician with the ECG.",
    "Factors": "Premature beats, electrode noise, atrial flutter, beats missed or added by the detector.",
    "Metrics": "AUC, sensitivity, specificity and positive predictive value per minute; alarm episodes and the error of the AF burden per record.",
    "Training data": f"{training['record'].nunique()} records of the AFDB, {len(training)} minutes",
    "Evaluation data": f"validation {v['record'].nunique()} records ({len(v)} minutes), test {test['record'].nunique()} records, opened once",
    "Quantitative analyses": f"validation AUC {roc_auc_score(v['af'], p_val):.3f}, sensitivity {(p_val[v['af'] == 1] >= t).mean():.3f}, specificity {(p_val[v['af'] == 0] < t).mean():.3f}",
    "Ethical considerations": "25 people of one hospital; performance by age, sex and in patients of Türkiye unknown.",
    "Caveats and recommendations": "Most false alarms come from premature beats; every alarm is judged on its strip.",
}
for section, text in card.items():
    print(f"{section}: {text}")

**Python note.** The model card is a dictionary from the nine sections of Mitchell and colleagues to their texts. The *f-strings* put computed values into the texts: `{roc_auc_score(...):.3f}` writes the AUC with three decimals, and `{digest[:12]}` the first twelve digits of the fingerprint. `nunique()` counts the different records of a table. In the quantitative analyses, `p_val[v['af'] == 1] >= t` keeps the AF minutes and asks which of them reach the threshold; the mean of True and False is the sensitivity.

**Observe and reflect.** The card says what the model is for and what it is not for. Which line of the card would you read first before you let the prototype run on your own ward, and which line is missing for your patients?

**Exercise F.** Make a copy of `model` with the threshold 0.38 instead of the threshold of the model, save it as `af_model_test_copy.json` and compute the probabilities of the validation minutes with the copy. How many validation minutes lose their alarm?

In [ ]:
data = exercise_table("F", p_val=p_val, validation=validation)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
copy_ = dict(model, threshold=0.38)
save_model(copy_, "af_model_test_copy.json")
p_copy = af_probability(copy_, validation)
answer = int(((p_val >= model["threshold"]) & (p_copy < copy_["threshold"])).sum())
teach.check("Exercise F", answer)

#### The report as HL7 FHIR resources

*Lecture page: [Introduction to the HL7 FHIR standard (conceptual)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#hl7-fhir-a-report-that-other-systems-can-read)*

The report of the prototype is a Bundle of four FHIR resources: a Device for the model, an Observation for the AF burden, a DiagnosticReport and a Provenance. It stays preliminary until a physician confirms or rejects the finding. The patient of this section is VC-0128 of the telemetry ward, whose 24 hours come from the course package.

**From the lecture to the code.** The status preliminary or final: `status`. The burden in the unit "%" of UCUM: `valueQuantity`. The code of AF in SNOMED CT, 49436004: `conclusionCode`. The author and the verifier: the agents of the Provenance. The checks: `check_bundle`.

In [ ]:
BASE_URL = "http://example.org/fhir"


def fhir_bundle(patient_id, result, model, digest, issued, decision=None, physician=None):
    """The report of one recording as HL7 FHIR R4 resources: the model as a Device, the AF burden as an Observation, a DiagnosticReport and a Provenance. The report stays preliminary until a physician confirms or rejects it."""
    status = "final" if decision in ("confirmed", "rejected") else "preliminary"
    device = {"resourceType": "Device", "id": "af-rr",
              "deviceName": [{"name": model["name"], "type": "model-name"}],
              "version": [{"value": model["version"]}],
              "note": [{"text": "SHA-256 " + digest}]}
    burden = {"resourceType": "Observation", "id": "af-burden", "status": status,
              "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
              "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF burden"},
              "subject": {"reference": "Patient/" + patient_id},
              "valueQuantity": {"value": result["burden_percent"], "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
              "device": {"reference": "Device/af-rr"}}
    if result.get("start"):
        burden["effectivePeriod"] = {"start": result["start"], "end": result["end"]}
    report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status,
              "code": {"text": "Holter recording, AF analysis"},
              "subject": {"reference": "Patient/" + patient_id}, "issued": issued,
              "result": [{"reference": "Observation/af-burden"}],
              "conclusion": result["text"]}
    agents = [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]
    if status == "final":
        report["performer"] = [{"reference": "Practitioner/" + physician}]
        agents.append({"type": {"text": "verifier"}, "who": {"reference": "Practitioner/" + physician}})
        if decision == "confirmed" and result["burden_percent"] > 0:
            report["conclusionCode"] = [{"coding": [{"system": "http://snomed.info/sct", "code": "49436004",
                                                      "display": "Atrial fibrillation (disorder)"}]}]
    provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                  "recorded": issued, "agent": agents}
    resources = [device, burden, report, provenance]
    return {"resourceType": "Bundle", "type": "collection",
            "entry": [{"fullUrl": f"{BASE_URL}/{r['resourceType']}/{r['id']}", "resource": r} for r in resources]}


def check_bundle(bundle):
    """Simple checks of a bundle: every resource has a type and an id, every reference inside the bundle resolves, and the burden has a UCUM unit. Patient and Practitioner live in the hospital system."""
    problems = []
    present = {f"{e['resource']['resourceType']}/{e['resource'].get('id')}" for e in bundle["entry"]}
    for e in bundle["entry"]:
        r = e["resource"]
        if not r.get("id"):
            problems.append(r["resourceType"] + ": no id")
        for ref in re.findall(r'"reference": "([^"]+)"', json.dumps(r)):
            if ref.split("/")[0] not in ("Patient", "Practitioner") and ref not in present:
                problems.append("reference does not resolve: " + ref)
        if r["resourceType"] == "Observation":
            q = r.get("valueQuantity", {})
            if q.get("system") != "http://unitsofmeasure.org":
                problems.append("Observation: the unit is not UCUM")
    return problems

rec_0128 = holter.recording()
t0128 = pd.DataFrame(minutes_of(rec_0128["detected_s"]))
t0128 = t0128.join(pd.DataFrame([features(x) for x in t0128["rr"]], index=t0128.index))
alarm_0128 = af_probability(model, t0128) >= model["threshold"]
runs_0128 = alarm_episodes(alarm_0128)
result = {"start": "2026-11-18T08:00:00+03:00", "end": "2026-11-19T08:00:00+03:00", "burden_percent": round(100 * float(alarm_0128.mean()), 1),
          "text": f"AF burden {100 * alarm_0128.mean():.1f} %, {len(runs_0128)} alarm episodes; model {model['version']}."}
bundle = fhir_bundle("VC-0128", result, model, digest, "2026-11-19T09:30:00+03:00")
print("problems:", check_bundle(bundle))
print(json.dumps(bundle["entry"][1]["resource"], indent=2, ensure_ascii=False))

**Python note.** `holter.recording()` gives the 24 hours of VC-0128: the true beats, the beats that the detector of the recorder found, and the episodes of AF. The prototype sees only `detected_s`, as on the ward. `fhir_bundle` builds the resources as dictionaries inside dictionaries, and `json.dumps(..., indent=2, ensure_ascii=False)` prints one of them as the JSON that another system would receive. `bundle["entry"][1]["resource"]` is the second entry of the bundle, the Observation. In `check_bundle`, `re.findall` finds every value of a key "reference" in the text of a resource.

In [ ]:
final = fhir_bundle("VC-0128", result, model, digest, "2026-11-19T09:45:00+03:00", decision="confirmed", physician="dr-demo")
report = final["entry"][2]["resource"]
print("status:", report["status"], "| performer:", report["performer"], "| code:", report["conclusionCode"][0]["coding"][0]["display"])
print("agents of the Provenance:", [a["type"]["text"] + " " + a["who"]["reference"] for a in final["entry"][3]["resource"]["agent"]])
print("problems:", check_bundle(final))

**Observe and reflect.** The preliminary report claims nothing about AF in its codes, and only the decision of the cardiologist adds the code 49436004. Who should be able to change the status back from final, and what should the record keep when that happens?

**Exercise G.** Count the references, the values of the keys "reference", in the final bundle `final`. How many are there?

In [ ]:
data = exercise_table("G", final=final)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(re.findall(r'"reference": "([^"]+)"', json.dumps(final)))
teach.check("Exercise G", answer)

#### The prediction service and the files of the prototype

*Lecture page: [Real-time prediction API (Flask/FastAPI)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#prediction-services-the-model-behind-an-address)*

A prediction service answers requests in JSON: The request carries the beat times of a recording, and the answer the result with the version and the fingerprint of the model. The service of this section listens only on this computer, at 127.0.0.1. The same service is then written with Flask, the answer for the beats of one minute is timed, and two cells write the files of the prototype, `af_cdss.py` with the functions of the notebook and `app_af.py` with the web interface; the lecture page describes the interface in its section on Streamlit. The section ends with the whole chain of the mini project: an ECG file of one hour of VC-0128 in, the prediction, the SHAP values of the first alarm minute and the FHIR report out.

**From the lecture to the code.** The answer of the service: `predict`. The address: `url`. The request: `payload`. The answer: `response`.

**Start cell.** For the section «The prediction service and the files of the prototype», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the section «The prediction service and the files of the prototype»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# functions of earlier parts, kept unchanged in the course package
from vclinic.week10 import features, choose_threshold, alarm_episodes

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
import hashlib
import json
import re
from sklearn.linear_model import LogisticRegression
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]
def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X
def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}
def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))
training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())
def contributions(model, row):
    """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
    x = design(pd.DataFrame([row]), model["features"]).iloc[0]
    z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
    return {name: model["coef"][name] * z[name] for name in model["features"]}
def save_model(model, path):
    """Write the model as JSON and return the SHA-256 of the file, its fingerprint."""
    text = json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False)
    pathlib.Path(path).write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()
digest = save_model(model, "af_model_v1.0.0.json")
BASE_URL = "http://example.org/fhir"
def fhir_bundle(patient_id, result, model, digest, issued, decision=None, physician=None):
    """The report of one recording as HL7 FHIR R4 resources: the model as a Device, the AF burden as an Observation, a DiagnosticReport and a Provenance. The report stays preliminary until a physician confirms or rejects it."""
    status = "final" if decision in ("confirmed", "rejected") else "preliminary"
    device = {"resourceType": "Device", "id": "af-rr",
              "deviceName": [{"name": model["name"], "type": "model-name"}],
              "version": [{"value": model["version"]}],
              "note": [{"text": "SHA-256 " + digest}]}
    burden = {"resourceType": "Observation", "id": "af-burden", "status": status,
              "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
              "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF burden"},
              "subject": {"reference": "Patient/" + patient_id},
              "valueQuantity": {"value": result["burden_percent"], "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
              "device": {"reference": "Device/af-rr"}}
    if result.get("start"):
        burden["effectivePeriod"] = {"start": result["start"], "end": result["end"]}
    report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status,
              "code": {"text": "Holter recording, AF analysis"},
              "subject": {"reference": "Patient/" + patient_id}, "issued": issued,
              "result": [{"reference": "Observation/af-burden"}],
              "conclusion": result["text"]}
    agents = [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]
    if status == "final":
        report["performer"] = [{"reference": "Practitioner/" + physician}]
        agents.append({"type": {"text": "verifier"}, "who": {"reference": "Practitioner/" + physician}})
        if decision == "confirmed" and result["burden_percent"] > 0:
            report["conclusionCode"] = [{"coding": [{"system": "http://snomed.info/sct", "code": "49436004",
                                                      "display": "Atrial fibrillation (disorder)"}]}]
    provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                  "recorded": issued, "agent": agents}
    resources = [device, burden, report, provenance]
    return {"resourceType": "Bundle", "type": "collection",
            "entry": [{"fullUrl": f"{BASE_URL}/{r['resourceType']}/{r['id']}", "resource": r} for r in resources]}
def check_bundle(bundle):
    """Simple checks of a bundle: every resource has a type and an id, every reference inside the bundle resolves, and the burden has a UCUM unit. Patient and Practitioner live in the hospital system."""
    problems = []
    present = {f"{e['resource']['resourceType']}/{e['resource'].get('id')}" for e in bundle["entry"]}
    for e in bundle["entry"]:
        r = e["resource"]
        if not r.get("id"):
            problems.append(r["resourceType"] + ": no id")
        for ref in re.findall(r'"reference": "([^"]+)"', json.dumps(r)):
            if ref.split("/")[0] not in ("Patient", "Practitioner") and ref not in present:
                problems.append("reference does not resolve: " + ref)
        if r["resourceType"] == "Observation":
            q = r.get("valueQuantity", {})
            if q.get("system") != "http://unitsofmeasure.org":
                problems.append("Observation: the unit is not UCUM")
    return problems
rec_0128 = holter.recording()

In [ ]:
def predict(request, model, digest):
    """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
    table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
    table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
    p = af_probability(model, table)
    alarm = p >= model["threshold"]
    runs = alarm_episodes(alarm)
    return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
            "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
            "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

answer_0128 = predict({"patient_id": "VC-0128", "beat_times_s": list(rec_0128["detected_s"])}, model, digest)
print({k: v for k, v in answer_0128.items() if k != "episodes"})

**Python note.** `predict` receives a dictionary, the request, and returns a dictionary, the answer: Both can travel as JSON. Inside, it uses the same functions as the notebook, `minutes_of`, `features`, `af_probability` and `alarm_episodes`, so the service and the notebook compute the same numbers. `int(...)` and `float(...)` turn the numbers of numpy into plain Python numbers, which JSON can write.

In [ ]:
import http.server
import threading
import urllib.request


class PredictionHandler(http.server.BaseHTTPRequestHandler):
    """Answers POST /predict with the result of predict() as JSON."""

    def do_POST(self):
        body = self.rfile.read(int(self.headers["Content-Length"]))
        answer = json.dumps(predict(json.loads(body), model, digest)).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(answer)

    def log_message(self, *args):
        pass


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
threading.Thread(target=server.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{server.server_port}/predict"

rec_0128 = holter.recording()
payload = {"patient_id": "VC-0128", "beat_times_s": [round(t, 3) for t in rec_0128["detected_s"]]}
request = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"})
local = urllib.request.build_opener(urllib.request.ProxyHandler({}))   # the request stays on this computer, past any proxy
response = json.loads(local.open(request, timeout=120).read().decode("utf-8"))
server.shutdown()

print(round(len(json.dumps(payload)) / 1024), "kB sent to", url)
print({k: v for k, v in response.items() if k != "episodes"})
print(pd.DataFrame(response["episodes"]))

**Python note.** `http.server.ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)` opens a small web server on this computer; the port 0 lets the operating system choose a free port, which `server.server_port` gives. The class `PredictionHandler` answers every POST request: It reads the body, turns it into a dictionary with `json.loads`, calls `predict` and writes the answer. `threading.Thread(..., daemon=True).start()` runs the server beside the notebook, so the next lines can send a request to it. `urllib.request.Request` builds the request with its body in bytes; `ProxyHandler({})` makes sure that it stays on this computer even when a proxy server is set, and `server.shutdown()` stops the server at the end.

**The same service with Flask.** A web framework takes over the work of the server: With Flask, one line above a function makes it the answer to POST /predict. The next cell writes the service again with Flask, sends the same request through the test client of Flask, which needs no network, and times the answer for the beats of one minute, as the central station would send them when a minute ends.

In [ ]:
import time

try:
    from flask import Flask, jsonify, request as flask_request
except ImportError:
    Flask = None
    print("Flask is not installed; install it with pip install flask.")

if Flask is not None:
    api = Flask("af_cdss")

    @api.post("/predict")
    def predict_route():
        """The same service with Flask: POST /predict with the beat times in JSON, the result back in JSON."""
        return jsonify(predict(flask_request.get_json(), model, digest))

if Flask is not None:
    client = api.test_client()
    reply = client.post("/predict", json=payload).get_json()
    print({k: v for k, v in reply.items() if k != "episodes"})
    print("Flask gives the same answer as the first service:", reply == response)
    one_minute = [t for t in payload["beat_times_s"] if 3600 <= t <= 3662]
    waits = []
    for _ in range(50):
        started = time.perf_counter()
        client.post("/predict", json={"patient_id": "VC-0128", "beat_times_s": one_minute})
        waits.append(1000 * (time.perf_counter() - started))
    print("Answer for the beats of one minute, median of 50 requests:", round(float(np.median(waits)), 1), "ms")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c197)*

**Python note.** `@api.post("/predict")` is a *decorator*: The line above the function registers it as the answer to POST requests at /predict. `flask_request.get_json()` reads the body of the request as a dictionary, and `jsonify` writes the answer as JSON. `api.test_client()` sends requests to the service inside the notebook, so no port is opened. `time.perf_counter()` reads a clock in seconds; the difference before and after a request, times 1000, is its time in milliseconds. To run the service for other programs, save it in a file and start it with `flask --app <file> run`; it then listens at http://127.0.0.1:5000.

**The two files of the prototype.** The next two cells write the files with the magic command `%%writefile`. The first holds the functions of the notebook, the second the web interface. To start the interface, open the Anaconda Prompt, go to the folder of this notebook with `cd`, and run `streamlit run app_af.py`. The browser opens the page at http://localhost:8501; the address stays on this computer. Stop the interface with Ctrl+C in the Anaconda Prompt.

In [ ]:
%%writefile af_cdss.py
"""af_cdss: the functions of the AF decision support of the course, moved from the Jupyter notebook into a file (a prototype, not for clinical use)."""
import hashlib
import json
import pathlib
import re

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

WINDOW_S = 60


def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows


def sample_entropy(rr, m=1, r=0.03):
    """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
    n = len(rr)

    def pairs(length):
        t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
        alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
        return (alike.sum() - len(t)) / 2

    b, a = pairs(m), pairs(m + 1)
    return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


def features(rr):
    """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
    d = np.diff(rr)
    return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
            "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]


def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X


def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


def choose_threshold(p, y, sensitivity=0.95):
    """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


def alarm_episodes(alarm):
    """Runs of consecutive alarm minutes, as (first, last) positions."""
    runs, first = [], None
    for k, a in enumerate(alarm):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    if first is not None:
        runs.append((first, len(alarm) - 1))
    return runs


def by_record(table, p, threshold):
    """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
    rows = []
    for record, part in table.assign(p=p).groupby("record", sort=True):
        alarm = (part["p"] >= threshold).to_numpy()
        runs = alarm_episodes(alarm)
        false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
        rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                     "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
    return pd.DataFrame(rows).set_index("record")


def contributions(model, row):
    """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
    x = design(pd.DataFrame([row]), model["features"]).iloc[0]
    z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
    return {name: model["coef"][name] * z[name] for name in model["features"]}


def save_model(model, path):
    """Write the model as JSON and return the SHA-256 of the file, its fingerprint."""
    text = json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False)
    pathlib.Path(path).write_text(text, encoding="utf-8")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_model(path):
    """Read a model written by save_model; returns the model and the SHA-256 of the file."""
    text = pathlib.Path(path).read_text(encoding="utf-8")
    return json.loads(text), hashlib.sha256(text.encode("utf-8")).hexdigest()

BASE_URL = "http://example.org/fhir"


def fhir_bundle(patient_id, result, model, digest, issued, decision=None, physician=None):
    """The report of one recording as HL7 FHIR R4 resources: the model as a Device, the AF burden as an Observation, a DiagnosticReport and a Provenance. The report stays preliminary until a physician confirms or rejects it."""
    status = "final" if decision in ("confirmed", "rejected") else "preliminary"
    device = {"resourceType": "Device", "id": "af-rr",
              "deviceName": [{"name": model["name"], "type": "model-name"}],
              "version": [{"value": model["version"]}],
              "note": [{"text": "SHA-256 " + digest}]}
    burden = {"resourceType": "Observation", "id": "af-burden", "status": status,
              "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "procedure"}]}],
              "code": {"coding": [{"system": "http://example.org/vclinic", "code": "af-burden"}], "text": "AF burden"},
              "subject": {"reference": "Patient/" + patient_id},
              "valueQuantity": {"value": result["burden_percent"], "unit": "%", "system": "http://unitsofmeasure.org", "code": "%"},
              "device": {"reference": "Device/af-rr"}}
    if result.get("start"):
        burden["effectivePeriod"] = {"start": result["start"], "end": result["end"]}
    report = {"resourceType": "DiagnosticReport", "id": "af-report", "status": status,
              "code": {"text": "Holter recording, AF analysis"},
              "subject": {"reference": "Patient/" + patient_id}, "issued": issued,
              "result": [{"reference": "Observation/af-burden"}],
              "conclusion": result["text"]}
    agents = [{"type": {"text": "author"}, "who": {"reference": "Device/af-rr"}}]
    if status == "final":
        report["performer"] = [{"reference": "Practitioner/" + physician}]
        agents.append({"type": {"text": "verifier"}, "who": {"reference": "Practitioner/" + physician}})
        if decision == "confirmed" and result["burden_percent"] > 0:
            report["conclusionCode"] = [{"coding": [{"system": "http://snomed.info/sct", "code": "49436004",
                                                      "display": "Atrial fibrillation (disorder)"}]}]
    provenance = {"resourceType": "Provenance", "id": "af-provenance", "target": [{"reference": "DiagnosticReport/af-report"}],
                  "recorded": issued, "agent": agents}
    resources = [device, burden, report, provenance]
    return {"resourceType": "Bundle", "type": "collection",
            "entry": [{"fullUrl": f"{BASE_URL}/{r['resourceType']}/{r['id']}", "resource": r} for r in resources]}


def check_bundle(bundle):
    """Simple checks of a bundle: every resource has a type and an id, every reference inside the bundle resolves, and the burden has a UCUM unit. Patient and Practitioner live in the hospital system."""
    problems = []
    present = {f"{e['resource']['resourceType']}/{e['resource'].get('id')}" for e in bundle["entry"]}
    for e in bundle["entry"]:
        r = e["resource"]
        if not r.get("id"):
            problems.append(r["resourceType"] + ": no id")
        for ref in re.findall(r'"reference": "([^"]+)"', json.dumps(r)):
            if ref.split("/")[0] not in ("Patient", "Practitioner") and ref not in present:
                problems.append("reference does not resolve: " + ref)
        if r["resourceType"] == "Observation":
            q = r.get("valueQuantity", {})
            if q.get("system") != "http://unitsofmeasure.org":
                problems.append("Observation: the unit is not UCUM")
    return problems


def predict(request, model, digest):
    """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
    table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
    table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
    p = af_probability(model, table)
    alarm = p >= model["threshold"]
    runs = alarm_episodes(alarm)
    return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
            "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
            "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}


def beats_from_ecg(signal, fs, start_s=0.0):
    """The beat times of one lead of ECG in seconds: the R peaks that the detector of Week 5 finds, plus the start of the recording."""
    from vclinic import beats
    return start_s + beats.r_peaks(np.asarray(signal, dtype=float), fs=fs) / fs

In [ ]:
%%writefile app_af.py
"""AF decision support: the web interface of the prototype of the course (not for clinical use). Start it in the Anaconda Prompt with: streamlit run app_af.py"""
import datetime
import json
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

HERE = pathlib.Path(__file__).resolve().parent
for folder in [HERE, *HERE.parents]:
    if (folder / "vclinic" / "__init__.py").exists():
        sys.path.insert(0, str(folder))
        break
sys.path.insert(0, str(HERE))
import af_cdss as cdss
from vclinic import afdb, holter

st.set_page_config(page_title="AF decision support", layout="wide")
st.title("AF decision support")
st.caption("A prototype of the course with synthetic and public data, not for clinical use.")

files = sorted(HERE.glob("af_model_v*.json"))
if not files:
    st.error("No model file next to the app. Run Section 6 of the Jupyter notebook first.")
    st.stop()
chosen = st.sidebar.selectbox("Model file", [f.name for f in files], index=len(files) - 1)
model, digest = cdss.load_model(HERE / chosen)
st.sidebar.write(f"Version {model['version']} · threshold {model['threshold']:.2f} · SHA-256 {digest[:12]}")


@st.cache_data
def ward_recordings():
    return holter.ward()


ward = ward_recordings()
sources = {f"Ward {pid}": ("ward", pid) for pid in ward}
sources.update({f"AFDB {r}": ("afdb", r) for r in afdb.RECORDS if (afdb.default_folder() / f"{r}.qrs").exists()})
label = st.sidebar.selectbox("Recording", list(sources))
kind, rid = sources[label]
beat_s = ward[rid]["detected_s"] if kind == "ward" else afdb.beats(rid)
upload = st.sidebar.file_uploader("Or upload an ECG: a CSV file with one lead in millivolts in a column named mV", type=["csv"])
if upload is not None:
    fs_upload = st.sidebar.number_input("Sampling rate of the ECG (Hz)", min_value=100, max_value=2000, value=250, step=50)
    rid = st.sidebar.text_input("Pseudonymous identifier of the patient", value="VC-0128")
    signal = pd.read_csv(upload)["mV"].to_numpy(dtype=float)
    beat_s = cdss.beats_from_ecg(signal, fs_upload)
    kind, label = "ecg", upload.name

table = pd.DataFrame(cdss.minutes_of(beat_s))
table = table.join(pd.DataFrame([cdss.features(x) for x in table["rr"]], index=table.index))
p = cdss.af_probability(model, table)
alarm = p >= model["threshold"]
runs = cdss.alarm_episodes(alarm)
hours = (table["start_s"] / 3600).round(3)

c1, c2, c3 = st.columns(3)
c1.metric("AF burden", f"{100 * alarm.mean():.1f} %")
c2.metric("Alarm episodes", len(runs))
c3.metric("Minutes analysed", len(table))
st.line_chart(pd.DataFrame({"AF probability": p, "threshold": model["threshold"]}, index=hours))

if runs:
    episodes = pd.DataFrame([{"start (h)": hours.iloc[a], "minutes": b - a + 1,
                              "highest probability": round(float(p[a:b + 1].max()), 2)} for a, b in runs])
    st.dataframe(episodes)
    k = st.selectbox("Alarm episode to review", list(range(len(runs))),
                     format_func=lambda i: f"{i + 1}: {episodes.iloc[i, 0]:.2f} h")
    row = table.iloc[runs[k][0]]
    left, right = st.columns(2)
    with left:
        st.write("SHAP values: what each feature adds to the log-odds of AF in the first minute of the episode")
        st.bar_chart(pd.Series(cdss.contributions(model, row), name="log-odds"))
    with right:
        fig, ax = plt.subplots(figsize=(6, 2.6))
        if kind == "afdb" and (afdb.default_folder() / f"{rid}.dat").exists():
            t, x = afdb.strip(rid, row["start_s"], 10)
            ax.plot(t - t[0], x[:, 0], lw=0.7)
            ax.set_title(f"AFDB {rid} · MIT-BIH AFDB, PhysioNet, ODC-By 1.0", fontsize=8)
        elif kind == "ecg":
            first_sample = int(row["start_s"] * fs_upload)
            part = signal[first_sample:first_sample + int(10 * fs_upload)]
            ax.plot(np.arange(len(part)) / fs_upload, part, lw=0.7)
            ax.set_title(f"{label} · uploaded ECG, the first ten seconds of the minute", fontsize=8)
        elif kind == "ward":
            t, x = holter.strip(ward[rid], row["start_s"])
            ax.plot(t, x, lw=0.7)
            ax.set_title(f"{rid} · lead II, synthetic", fontsize=8)
        else:
            ax.plot(row["rr"], marker=".", lw=0.7)
            ax.set_title("RR intervals of the minute (s); no signal file", fontsize=8)
        ax.set_xlabel("s")
        st.pyplot(fig)
    review = f"alarm episode {k + 1}"
else:
    review = "no alarm"

st.subheader("Decision of the physician")
decision = st.radio("The AF finding of the prototype is", ["not reviewed", "confirmed", "rejected"])
reason = st.text_input("Reason (for example: premature beats, noise)")
physician = st.text_input("Physician", value="dr-demo")
code = {"not reviewed": None, "confirmed": "confirmed", "rejected": "rejected"}[decision]
now = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
if st.button("Save the decision in the log"):
    log = HERE / "decision_log.csv"
    entry = {"time": now, "recording": label, "review": review, "model_version": model["version"], "sha256": digest,
             "ai_burden_percent": round(100 * float(alarm.mean()), 1), "decision": code or "not reviewed", "reason": reason, "physician": physician}
    pd.DataFrame([entry]).to_csv(log, mode="a", header=not log.exists(), index=False)
    st.success("Saved in " + log.name)

period = ("2026-11-18T08:00:00+03:00", "2026-11-19T08:00:00+03:00") if kind == "ward" else (None, None)
result = {"start": period[0], "end": period[1], "burden_percent": round(100 * float(alarm.mean()), 1),
          "text": f"AF burden {100 * alarm.mean():.1f} %, {len(runs)} alarm episodes; model {model['version']}."}
bundle = cdss.fhir_bundle(rid, result, model, digest, now, code, physician)
st.download_button("Report as HL7 FHIR (JSON)", json.dumps(bundle, indent=2, ensure_ascii=False),
                   file_name=f"report_{rid}.json", mime="application/json")

In [ ]:
import importlib
import af_cdss
importlib.reload(af_cdss)
same = np.allclose(af_cdss.af_probability(model, validation), p_val)
print("af_cdss gives the same probabilities as the notebook:", same)
for name in ("af_cdss.py", "app_af.py"):
    print(name, pathlib.Path(name).stat().st_size, "bytes")

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c202)*

**The mini project from end to end.** The chain of the mini project runs on a file in the format that the web interface accepts for an upload: one lead of ECG in millivolts, in a column named mV. The first cell writes one hour of lead II of VC-0128, from 13:00 to 14:00, at 250 Hz. The second cell takes the four steps of the mini project: The detector of Week 5 finds the beats of the uploaded ECG, the model predicts the AF of every minute, SHAP explains the first alarm minute, and a FHIR report is produced. In the web interface, the same file can be uploaded in the side bar.

In [ ]:
def beats_from_ecg(signal, fs, start_s=0.0):
    """The beat times of one lead of ECG in seconds: the R peaks that the detector of Week 5 finds, plus the start of the recording."""
    from vclinic import beats
    return start_s + beats.r_peaks(np.asarray(signal, dtype=float), fs=fs) / fs


ecg_start_s = 5 * 3600.0   # 13:00, five hours after the start of the recording at 08:00
t_ecg, ecg_signal = holter.strip(rec_0128, ecg_start_s, duration_s=3600, fs=250)
ecg_file = "ecg_VC-0128_13-14.csv"
pd.DataFrame({"mV": np.round(ecg_signal, 4)}).to_csv(ecg_file, index=False)
print(ecg_file, len(ecg_signal), "values,", round(pathlib.Path(ecg_file).stat().st_size / 1e6, 1), "MB")

In [ ]:
signal = pd.read_csv(ecg_file)["mV"].to_numpy()                                   # 1. an ECG is uploaded
ecg_beat_s = beats_from_ecg(signal, fs=250, start_s=ecg_start_s)
truth = rec_0128["beat_s"][(rec_0128["beat_s"] >= ecg_start_s) & (rec_0128["beat_s"] < ecg_start_s + 3600)]
j = np.clip(np.searchsorted(truth, ecg_beat_s), 1, len(truth) - 1)
nearest = np.minimum(np.abs(ecg_beat_s - truth[j - 1]), np.abs(ecg_beat_s - truth[j]))
print(len(ecg_beat_s), "beats found,", int((nearest <= 0.05).sum()), "of them within 50 ms of a beat of the recording")

ecg_answer = predict({"patient_id": "VC-0128", "beat_times_s": list(ecg_beat_s)}, model, digest)   # 2. the model predicts
print({k: v for k, v in ecg_answer.items() if k not in ("episodes", "sha256")})
print([(holter.clock_text(e["start_s"]), e["minutes"]) for e in ecg_answer["episodes"]])

ecg_minutes = pd.DataFrame(minutes_of(ecg_beat_s))
ecg_minutes = ecg_minutes.join(pd.DataFrame([features(x) for x in ecg_minutes["rr"]], index=ecg_minutes.index))
ecg_p = af_probability(model, ecg_minutes)
first = int(np.argmax(ecg_p >= model["threshold"]))
ecg_shap = contributions(model, ecg_minutes.iloc[first])                           # 3. SHAP explains the first alarm minute
print("first alarm minute", holter.clock_text(ecg_minutes["start_s"].iloc[first]), "· probability", round(float(ecg_p[first]), 3),
      "· SHAP", {k: round(float(v), 2) for k, v in ecg_shap.items()})

ecg_result = {"start": "2026-11-18T13:00:00+03:00", "end": "2026-11-18T14:00:00+03:00", "burden_percent": ecg_answer["burden_percent"],
              "text": f"AF burden {ecg_answer['burden_percent']:.1f} %, {len(ecg_answer['episodes'])} alarm episodes; model {model['version']}."}
ecg_report = fhir_bundle("VC-0128", ecg_result, model, digest, "2026-11-18T14:01:00+03:00")   # 4. a report is produced
print([e["resource"]["resourceType"] for e in ecg_report["entry"]], "problems:", check_bundle(ecg_report))

↩ *[Back to where you were on the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#nbref-c205)*

**Python note.** `holter.strip` draws lead II of the synthetic recording from the times of its beats, with P waves in sinus rhythm and fibrillatory waves in AF. `beats_from_ecg` calls `r_peaks` of the course package, the detector of Week 5, and turns the positions of the peaks into seconds. `np.searchsorted` finds, for every beat found, its place among the beats of the recording, so that the distance to the nearest one can be measured. The four commented lines are the four steps of the mini project; `predict` is the function of the prediction service.

**Observe and reflect.** The interface reads the model from a file and shows its fingerprint in the side bar. What should happen on the ward when someone replaces the file with a new version during a shift?

**Exercise H.** How many minutes did the service analyse in the 24 hours of VC-0128?

In [ ]:
data = exercise_table("H", rec_0128=rec_0128)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(response["minutes"])
teach.check("Exercise H", answer)

**Stop 2 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#last-two-hours-deployment-and-publishing) and continue reading at «Last two hours: Deployment and publishing». At the next practice stop, the lecture page brings you back to this Jupyter notebook.

## Stop 3: The user test and version 1.1

*Lecture page: [Practice stop 3](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#stop-3)*

This stop runs the user test and improves the prototype. The alarm episodes of the four patients of the telemetry ward show what a physician would review in 24 hours, version 1.1 adds the sample entropy and learns again on the training records, and the test records judge both versions once.

**At the end of this stop:** The false alarm episodes of the ward in 24 hours (76), the false alarm episodes of version 1.1 on the validation records (69) and the sensitivity of version 1.1 on the test records (0.924).

The stop has three parts: **Python basics**, **Prompt and application** and **Ready application**. Work through them in this order. If you know Python, you may start with the prompts; to see only the result, start with the ready application. Every part opens with its start cell.

### Stop 3 · Python basics

*Lecture page: [User test session: Improving the CDSS prototype with feedback](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#user-test-session-improving-the-cdss-prototype-with-feedback)*

The ready application of this stop finds the alarm episodes of the patients of a telemetry ward, writes the decision of the cardiologist for each into a log, makes version 1.1 with a fifth number and compares both versions on the same test records. The examples below introduce these constructs on a few minutes: runs of alarm minutes, a log built row by row, a new list from an old one and the share of alarms among the minutes with AF.

Run the cells in order. The *Python note* under an example explains the new construct; exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells.

**Start cell.** For «Stop 3 · Python basics», the cell below repeats the setup of the course package and of the packages of the week, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Python basics»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

In [ ]:
# Python practice: a small example for learning the language
alarm = np.array([0, 1, 1, 0, 0, 1, 0, 1, 1, 1])   # 1: the minute raised an alarm
edges = np.diff(np.r_[0, alarm, 0])
first, after = np.where(edges == 1)[0], np.where(edges == -1)[0]
print(list(zip(first.tolist(), (after - 1).tolist())))

**Python note.** `np.r_[0, alarm, 0]` adds a 0 before and after the minutes, and `np.diff` marks every change: 1 where a run of alarm minutes starts and −1 in the minute after it ends. `np.where` finds these positions, so the runs are the minutes 1 to 2, 5 and 7 to 9: three alarm episodes. The function `alarm_episodes` of the ready application finds the same runs with a loop over the minutes.

In [ ]:
# Python practice: a small example for learning the language
rows = []
for patient, length, true_af in [("VC-0347", 3, 0), ("VC-0347", 12, 1), ("VC-0733", 2, 0)]:
    rows.append({"patient": patient, "minutes": length, "decision": "confirmed" if true_af else "rejected"})
log = pd.DataFrame(rows)
print(log)
print(log.groupby("decision").size())

**Python note.** A log grows one row at a time: Every pass of the loop appends a dictionary to the list `rows`, and `pd.DataFrame(rows)` turns the list into a table whose columns are the keys of the dictionaries. `groupby("decision").size()` then counts the rows of every decision. The ready application writes the decision log of the telemetry ward in this way, with the version and the fingerprint of the model in every row, and saves it as a CSV file.

In [ ]:
# Python practice: a small example for learning the language
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
V1_1 = V1_0 + ["sampen"]
print(V1_0, len(V1_0))
print(V1_1, len(V1_1))

**Python note.** `V1_0 + ["sampen"]` makes a new list with five names and leaves the list of version 1.0 as it was, so both versions can still be built and compared. `V1_0.append("sampen")` would instead change the old list itself, and version 1.0 would lose its own features. The ready application builds version 1.1 from the features of version 1.0 in this way.

In [ ]:
# Python practice: a small example for learning the language
af = np.array([1, 1, 1, 0, 0, 1, 0, 0])               # 1: an AF minute in the annotation
alarm = np.array([1, 0, 1, 0, 1, 1, 0, 0]) == 1
print("sensitivity", alarm[af == 1].mean(), "| specificity", 1 - alarm[af == 0].mean())

**Python note.** `alarm[af == 1]` keeps the alarms of the minutes with AF, and their mean is the share of AF minutes with an alarm, the sensitivity, here 0.75. `alarm[af == 0].mean()` is the share of false alarms among the minutes without AF, and 1 minus it is the specificity. The ready application judges version 1.0 and version 1.1 with the same two lines on the test records.

**Exercise 3.1.** Eleven consecutive minutes raise the alarms 1, 1, 0, 1, 0, 0, 1, 1, 1, 0 and 1. How many alarm episodes, runs of consecutive alarm minutes, do they hold?

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
edges = np.diff(np.r_[0, np.array([1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1]), 0])
answer = int((edges == 1).sum())
teach.check("Exercise 3.1", answer)

**Exercise 3.2.** Eight minutes have the annotation 1, 1, 1, 0, 0, 1, 0 and 1, where 1 is AF, and the alarms 1, 0, 1, 0, 1, 1, 0 and 1. What is the sensitivity of the alarms?

In [ ]:
answer = None   # your answer
teach.check("Exercise 3.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
af = np.array([1, 1, 1, 0, 0, 1, 0, 1])
alarm = np.array([1, 0, 1, 0, 1, 1, 0, 1]) == 1
answer = float(alarm[af == 1].mean())
teach.check("Exercise 3.2", answer)

**«Stop 3 · Python basics» is complete.** Next comes «Stop 3 · Prompt and application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#ai-pair-programming-part-10-reviewing-a-prototype) and continue reading at «Working with an AI assistant, part 10: Reviewing a prototype».

### Stop 3 · Prompt and application

This part has three prompts: «The alarm episodes of the ward», «Version 1.1 with the sample entropy» and «Both versions on the test records». The repair prompt and the prompt for an error message are in the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-10/PROMPTS.md) and in the prompts of Stop 1.

Every prompt has four steps. Copy the prompt and send it to your AI assistant. Run the data cell under it: It prepares the table "data" that the prompt describes and shows it. Paste the code of the assistant into the empty cell and run it. Then run the check cell, which prints *correct* or *not yet*. If the code of the assistant does not pass the check, run the cell *Ready code* and go on.

**Start cell.** For «Stop 3 · Prompt and application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Prompt and application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

from vclinic import afdb, holter
import hashlib
import json
from sklearn.linear_model import LogisticRegression

_SETS = {"training": "training", "validation": "validation", "test": "test"}


def _course():
    """The minutes of the 25 records of the AFDB and of the four patients of the telemetry ward, with their RR intervals, their five numbers and the probabilities of version 1.0 (computed once, about twenty seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "AF from RR intervals", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """The probability of AF for every minute of a table."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Runs of consecutive alarm minutes, as (first, last) positions."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """A table of the minutes of the AFDB with the chosen columns."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"record": m["record"], "set": m["set"].map(_SETS), "start (s)": m["start_s"].round(2),
            "AF": m["af"], "AF share": m["af_share"].round(3), "RR intervals (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "heart rate": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "probability": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)


_c = _course()
print(len(_c["minutes"]), "minutes of the 25 records of the AFDB")

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.metrics import roc_auc_score, roc_curve

#### Prompt 3.1: The alarm episodes of the ward

**Goal.** Count the alarm episodes that the physician of the ward would review in 24 hours, and the false ones among them.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The user test asks how much work the prototype gives the physician of the ward, and where its false alarms come from.  
> Data: The notebook holds a table named "data" with the minutes of the 24-hour recordings of the four patients of the telemetry ward of the virtual clinic, a synthetic data set: VC-0128 (paroxysmal atrial fibrillation, premature atrial beats and two periods of electrode noise), VC-0347 (sinus rhythm with frequent premature atrial beats, no AF), VC-0519 (persistent AF) and VC-0733 (sinus rhythm with long periods of electrode noise, no AF). Each row is a minute: "patient" gives the patient, "start (h)" the start of the minute in hours from the start of the recording, "AF" is 1 when more than half of the minute lies in AF in the truth of the recording, and "probability" is the probability of AF from version 1.0 of a model of the course. The model raises an alarm when the probability is at least 0.37.  
> Task: Within each patient, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm, and a false alarm episode is an episode in which no minute has AF. For each patient, count the alarm minutes, the false alarm minutes (alarm minutes without AF), the alarm episodes and the false alarm episodes.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each patient, print the total of the alarm episodes and of the false ones, and keep the total number of false alarm episodes under the name "answer".  
> Test: A patient without any AF minute can have no true alarm episode, and the false alarm episodes cannot outnumber the alarm episodes.  

In [ ]:
# Data of Prompt 3.1: the table "data" that the prompt describes
_w = _course()["ward_minutes"]
data = pd.DataFrame({"patient": _w["record"], "start (h)": (_w["start_s"] / 3600).round(4), "AF": _w["af"],
                     "probability": _w["p10"].round(4)}).reset_index(drop=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.1
teach.check_step("Prompt 3.1", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.1 (run it only if the code of the assistant does not pass the check)
_w = _course()["ward_minutes"]
data = pd.DataFrame({"patient": _w["record"], "start (h)": (_w["start_s"] / 3600).round(4), "AF": _w["af"],
                     "probability": _w["p10"].round(4)}).reset_index(drop=True)


def episodes(alarm, af):
    runs, first = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    return len(runs), sum(1 for s, e in runs if af[s:e + 1].sum() == 0)


d = data.sort_values(["patient", "start (h)"])
threshold = 0.37
rows = []
for patient, part in d.groupby("patient"):
    alarm = part["probability"].to_numpy() >= threshold
    n, false = episodes(alarm, part["AF"].to_numpy())
    rows.append({"patient": patient, "alarm minutes": int(alarm.sum()),
                         "false alarm minutes": int((alarm & (part["AF"].to_numpy() == 0)).sum()),
                         "alarm episodes": n, "false alarm episodes": false})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
print("to review in 24 hours:", int(table["alarm episodes"].sum()), "episodes, false:",
      int(table["false alarm episodes"].sum()))
answer = int(table["false alarm episodes"].sum())
teach.check_step("Prompt 3.1", "answer", globals())

**What the result shows.** The physician would review 76 false alarm episodes in 24 hours, almost all of them in VC-0347, whose premature atrial beats look irregular to the four numbers, and in VC-0733, whose recorder misses and adds beats in the noise.

#### Prompt 3.2: Version 1.1 with the sample entropy

**Goal.** Add the sample entropy as a fifth number, fit version 1.1 on the training records and count its false alarm episodes on the validation records.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The user test showed that premature beats raise most false alarms. Premature beats repeat the same short and long intervals, while AF has no pattern, so a number that measures the pattern may help. The test records must stay untouched.  
> Data: The notebook holds a table named "data" with the training and validation minutes of the MIT-BIH Atrial Fibrillation Database (AFDB), 25 Holter recordings of about ten hours of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute; the test records are not in it. Each row is a minute: "record" gives the recording, "set" its set ("training", 15 records; "validation", 5 records) and "start (s)" the start of the minute in seconds. "AF" is 1 when more than half of the minute lies in AF as marked by the experts. Each cell of "RR intervals (s)" holds the RR intervals of the minute in seconds, in their order. The four numbers of the minute are "heart rate" (per minute), "CV", "RMSSD/RR" and "pNN50", as in the earlier steps.  
> Task: For every minute, also compute the sample entropy of its RR intervals with stretches of one interval and a tolerance of 0.03 seconds: Of the first n − 1 intervals of the minute, count the pairs that differ by at most 0.03 seconds (B), and count how many of these pairs are still alike when each interval is followed by the next one (A). The sample entropy is the natural logarithm of B divided by A; count A as 1 when it is 0, and give the minute the natural logarithm of n when B is 0. Fit a logistic regression for AF on the training minutes with at most 3,000 iterations, on five inputs: the heart rate, the natural logarithm of the CV, the natural logarithm of the RMSSD/RR, the pNN50 and the sample entropy, each turned into a z-score with the mean and the standard deviation of the training minutes. Choose the threshold on the validation minutes only: Among 0.05, 0.06, 0.07 and so on up to 0.95, take the highest value at which at least 95 percent of the validation minutes with AF have a probability at or above it. Then, within each validation record, sort the minutes by their start, find the alarm episodes, runs of consecutive alarm minutes, and count the false alarm episodes, in which no minute has AF.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Print the validation AUC, the threshold, the number of false alarm episodes and the median sample entropy of the training minutes with and without AF, and keep the number of false alarm episodes under the name "answer".  
> Test: No test minute may be used, and the sample entropy of every minute must be 0 or more.  

In [ ]:
# Data of Prompt 3.2: the table "data" that the prompt describes
data = _minutes_table(["record", "set", "start (s)", "AF", "RR intervals (s)",
                       "heart rate", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.2
teach.check_step("Prompt 3.2", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.2 (run it only if the code of the assistant does not pass the check)
data = _minutes_table(["record", "set", "start (s)", "AF", "RR intervals (s)",
                       "heart rate", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))


def episodes(alarm, af):
    runs, first = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    return len(runs), sum(1 for s, e in runs if af[s:e + 1].sum() == 0)



def sampen(rr, r=0.03):
    rr = np.asarray(rr, dtype=float)
    n = len(rr)
    a1 = rr[: n - 1]
    alike = np.abs(a1[:, None] - a1[None, :]) <= r
    next_ = np.abs(rr[1:][:, None] - rr[1:][None, :]) <= r
    B = (alike.sum() - len(a1)) / 2
    A = ((alike & next_).sum() - len(a1)) / 2
    return float(np.log(B / max(A, 1))) if B > 0 else float(np.log(n))


d = data.copy()
d["sample entropy"] = d["RR intervals (s)"].apply(sampen)
X = pd.DataFrame({"heart rate": d["heart rate"], "log CV": np.log(d["CV"]),
                    "log RMSSD/RR": np.log(d["RMSSD/RR"]), "pNN50": d["pNN50"], "SampEn": d["sample entropy"]})
tr_ = (d["set"] == "training").to_numpy()
va_ = ~tr_
mean, sd = X[tr_].mean(), X[tr_].std()
lr = LogisticRegression(max_iter=3000).fit((X[tr_] - mean) / sd, d["AF"][tr_])
p = lr.predict_proba((X[va_] - mean) / sd)[:, 1]
y = d["AF"].to_numpy()[va_]
threshold = max(t for t in np.round(np.arange(0.05, 0.96, 0.01), 2) if (p[y == 1] >= t).mean() >= 0.95)
v = d[va_].assign(p_=p).sort_values(["record", "start (s)"])
false_total = 0
for record, part in v.groupby("record"):
    false_total += episodes(part["p_"].to_numpy() >= threshold, part["AF"].to_numpy())[1]
print("validation AUC", round(roc_auc_score(y, p), 3), "| threshold", threshold,
      "| false alarm episodes", false_total)
print("median sample entropy, training: AF", round(d[tr_ & (d["AF"] == 1)]["sample entropy"].median(), 2),
      "| other minutes", round(d[tr_ & (d["AF"] == 0)]["sample entropy"].median(), 2))
answer = int(false_total)
teach.check_step("Prompt 3.2", "answer", globals())

**What the result shows.** Version 1.1 raises 69 false alarm episodes on the validation records, against 99 for version 1.0. The median sample entropy of the AF minutes lies far above that of the other minutes.

#### Prompt 3.3: Both versions on the test records

**Goal.** Open the test records once and compare the two versions of the model.

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: Before the clinic chooses a version for the trial of the ward, the cardiologist wants both versions judged on records that neither of them has seen, with the gains and the costs of version 1.1.  
> Data: The notebook holds a table named "data" with the minutes of the 5 test records of the MIT-BIH Atrial Fibrillation Database, Holter recordings of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. Each row is a minute: "record" gives the recording and "start (s)" the start of the minute in seconds. "AF" is 1 when more than half of the minute lies in AF as marked by the experts, and "AF share" is the share of the minute in AF, from 0 to 1. "probability, version 1.0" and "probability, version 1.1" are the probabilities of AF from two versions of a model of the course. Version 1.0 raises an alarm when its probability is at least 0.37, version 1.1 when its probability is at least 0.28.  
> Task: Within each record, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm, and a false alarm episode is an episode in which no minute has AF. For each version, compute on all test minutes the AUC, the sensitivity (the share of AF minutes with an alarm) and the specificity (the share of the other minutes without an alarm). Count the false alarm episodes over the test records, and compute the error of the AF burden: for each record, the absolute difference between 100 times the share of its minutes with an alarm and 100 times its mean AF share, averaged over the records.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each version, and keep the sensitivity of version 1.1, with three decimals, under the name "answer".  
> Test: Every AUC, sensitivity and specificity must lie between 0 and 1, and each threshold must be the one chosen on the validation records.  

In [ ]:
# Data of Prompt 3.3: the table "data" that the prompt describes
_c = _course()
_m = _c["minutes"]
_tr, _te = _m[_m["set"] == "training"], _m[_m["set"] == "test"]
_V11 = _c["model"]["features"] + ["sampen"]
_X = _c["design"](_tr, _V11)
_mu, _sd = _X.mean(), _X.std()
_lr = LogisticRegression(max_iter=3000).fit((_X - _mu) / _sd, _tr["af"])
_z = (_c["design"](_te, _V11) - _mu) / _sd
data = pd.DataFrame({"record": _te["record"], "start (s)": _te["start_s"].round(2), "AF": _te["af"],
                     "AF share": _te["af_share"].round(3), "probability, version 1.0": _te["p10"].round(4),
                     "probability, version 1.1": _lr.predict_proba(_z)[:, 1].round(4)}).reset_index(drop=True)
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

In [ ]:
# Check of Prompt 3.3
teach.check_step("Prompt 3.3", "answer", globals())

In [ ]:
#@title Ready code of Prompt 3.3 (run it only if the code of the assistant does not pass the check)
_c = _course()
_m = _c["minutes"]
_tr, _te = _m[_m["set"] == "training"], _m[_m["set"] == "test"]
_V11 = _c["model"]["features"] + ["sampen"]
_X = _c["design"](_tr, _V11)
_mu, _sd = _X.mean(), _X.std()
_lr = LogisticRegression(max_iter=3000).fit((_X - _mu) / _sd, _tr["af"])
_z = (_c["design"](_te, _V11) - _mu) / _sd
data = pd.DataFrame({"record": _te["record"], "start (s)": _te["start_s"].round(2), "AF": _te["af"],
                     "AF share": _te["af_share"].round(3), "probability, version 1.0": _te["p10"].round(4),
                     "probability, version 1.1": _lr.predict_proba(_z)[:, 1].round(4)}).reset_index(drop=True)


def episodes(alarm, af):
    runs, first = [], None
    for k, a in enumerate(list(alarm) + [False]):
        if a and first is None:
            first = k
        if not a and first is not None:
            runs.append((first, k - 1))
            first = None
    return len(runs), sum(1 for s, e in runs if af[s:e + 1].sum() == 0)


d = data.sort_values(["record", "start (s)"])
rows = []
for name, column, threshold in (("1.0", "probability, version 1.0", 0.37),
                                              ("1.1", "probability, version 1.1", 0.28)):
    p, y = d[column].to_numpy(), d["AF"].to_numpy()
    alarm = p >= threshold
    false, errors = 0, []
    for record, part in d.groupby("record"):
        a = part[column].to_numpy() >= threshold
        false += episodes(a, part["AF"].to_numpy())[1]
        errors.append(abs(100 * a.mean() - 100 * part["AF share"].mean()))
    rows.append({"version": name, "AUC": round(roc_auc_score(y, p), 3),
                         "sensitivity": round((alarm & (y == 1)).sum() / (y == 1).sum(), 3),
                         "specificity": round((~alarm & (y == 0)).sum() / (y == 0).sum(), 3),
                         "false alarm episodes": false, "error of the AF burden (points)": round(np.mean(errors), 1)})
table = pd.DataFrame(rows)
print(table.to_string(index=False))
answer = float(table.iloc[1]["sensitivity"])
teach.check_step("Prompt 3.3", "answer", globals())

**What the result shows.** Version 1.1 raises fewer false alarm episodes on the test records, but its sensitivity falls to 0.924. The change log of version 1.1 has to report both.

**«Stop 3 · Prompt and application» is complete.** Next comes «Stop 3 · Ready application», right below. To leave the stop here, [go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#ai-pair-programming-part-10-reviewing-a-prototype) and continue reading at «Working with an AI assistant, part 10: Reviewing a prototype».

### Stop 3 · Ready application

This part runs the user test of the prototype on the patients of a telemetry ward with a decision log, builds version 1.1 with the sample entropy as a fifth number and compares both versions on the test records.

Run the cells from top to bottom. *Python note* cells explain the code, and *Observe and reflect* cells ask questions about the results.

**Start cell.** For «Stop 3 · Ready application», the cell below repeats the setup and the data calls and code of the earlier cells that the cells after it use, so that the code runs without errors. The functions written in «Stop 1 · Ready application» and «Stop 2 · Ready application» come from the course package, which keeps them unchanged; their code is in that part. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: «Stop 3 · Ready application»
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

# functions of earlier parts, kept unchanged in the course package
from vclinic.week10 import features, choose_threshold, alarm_episodes, by_record, save_model

# the tables of the exercise prompts (Part F of the interactive lab)
from vclinic.week10 import exercise_table

# repeated from the earlier cells: the data and the code that the cells below use
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
WINDOW_S = 60
def minutes_of(beat_s, episodes=None, record=""):
    """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
    rr = np.diff(beat_s)
    rr_end = beat_s[1:]
    rows = []
    for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
        x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
        if len(x) < 20 or x.max() > 3.0:
            continue
        row = {"record": record, "start_s": float(start), "rr": x}
        if episodes is not None:
            af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
            row["af_share"] = af_s / WINDOW_S
        rows.append(row)
    return rows
minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
minutes["set"] = minutes["record"].map(SPLIT)
minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))
V1_0 = ["hr", "cv", "rmssd", "pnn50"]
LOGGED = ["cv", "rmssd"]
def design(table, names):
    """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
    X = table[names].astype(float).copy()
    for name in names:
        if name in LOGGED:
            X[name] = np.log(X[name])
    return X
def fit_model(names, version, training):
    """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
    X = design(training, names)
    mean, sd = X.mean(), X.std()
    lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
    return {"name": "AF from RR intervals", "version": version, "features": names,
            "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
            "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}
def af_probability(model, table):
    """The probability of AF for every minute of a table."""
    names = model["features"]
    z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
    w = np.array([model["coef"][name] for name in names])
    return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))
training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
model = fit_model(V1_0, "1.0.0", training)
p_val = af_probability(model, validation)
model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())
records_val = by_record(validation, p_val, model["threshold"])
digest = save_model(model, "af_model_v1.0.0.json")

# Python practice: a small example for learning the language
V1_0 = ["hr", "cv", "rmssd", "pnn50"]

#### The user test on the telemetry ward

*Lecture page: [The user test: Watching users at work](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#the-user-test-watching-users-at-work)*

The telemetry ward of the virtual clinic has four patients: VC-0128 with paroxysmal AF, premature atrial beats and two short periods of noise; VC-0347 with frequent premature atrial beats and no AF; VC-0519 in AF for the whole day; and VC-0733 with long periods of electrode noise and no AF. Their recordings come from the course package, and the prototype sees the beats that the recorder found. This section counts what the physician would review in 24 hours and writes a decision log.

**From the lecture to the code.** The minutes of the ward: `ward_minutes`. The alarm episodes of each patient: `by_record`. The decision log: `log`, saved as `decision_log_ward.csv`.

In [ ]:
ward = holter.ward()
ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                             for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))

ward_minutes["p10"] = af_probability(model, ward_minutes)
review = by_record(ward_minutes, ward_minutes["p10"].to_numpy(), model["threshold"])
print(review.round(1))
print("to review in 24 hours:", int(review["episodes"].sum()), "episodes, false:", int(review["false_episodes"].sum()))

**Python note.** `holter.ward()` returns a dictionary from the identifier of each patient to its recording. The list comprehension of `ward_minutes` cuts every recording into minutes with the same `minutes_of`, using the AF episodes of the recording as the truth; in the course they play the part of the reading of the strips by a cardiologist. `by_record` works on any table with a column "record", here the patients.

In [ ]:
rows = []
for pid, part in ward_minutes.groupby("record"):
    alarm = (part["p10"] >= model["threshold"]).to_numpy()
    for a, b in alarm_episodes(alarm):
        true_af = int(part["af"].to_numpy()[a:b + 1].sum() > 0)
        rows.append({"patient": pid, "start (h)": round(part["start_s"].iloc[a] / 3600, 2), "minutes": b - a + 1,
                     "decision": "confirmed" if true_af else "rejected",
                     "reason": "irregular RR without P waves" if true_af else
                                         ("electrode noise" if pid == "VC-0733" else "premature atrial beats"),
                     "version": model["version"], "SHA-256": digest[:12]})
log = pd.DataFrame(rows)
log.to_csv("decision_log_ward.csv", index=False)
print(log.groupby(["patient", "decision"]).size())
print(log.head(8).to_string(index=False))

**Python note.** For every alarm episode, the loop writes one row of the decision log: the patient, the start, the length, the decision, its reason, and the version and the fingerprint of the model. Here the decisions follow the truth of the synthetic recordings, so the log shows what a careful physician would decide; on a ward, the physician fills it in. `to_csv(..., index=False)` saves the table as a file of comma-separated values that a spreadsheet opens, and `groupby([...]).size()` counts the rows of each patient and decision.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9.6, 7.0), sharex=True)
for ax, (pid, part) in zip(axes, ward_minutes.groupby("record")):
    hours_ = part["start_s"] / 3600
    ax.fill_between(hours_, 0, part["af"], color="#C8C0B8", step="post")
    ax.plot(hours_, part["p10"], color="#9B1B1B", lw=0.7)
    ax.axhline(model["threshold"], color="#1F5F8B", ls="--", lw=0.8)
    ax.set_ylabel(pid, fontsize=9)
    ax.set_ylim(0, 1.02)
axes[-1].set_xlabel("hours from 08:00")
axes[0].set_title("Probability of AF from version 1.0 on the telemetry ward (grey: AF of the recording; synthetic data)", fontsize=10)
plt.tight_layout()
plt.show()

**Observe and reflect.** VC-0347 gives many short false alarms in the afternoon and evening, and VC-0733 some in the periods of noise. Which of the two causes could a better number of the minute remove, and which needs a check of the signal instead?

**Exercise I.** How many false alarm episodes does version 1.0 raise for VC-0347 in 24 hours?

In [ ]:
data = exercise_table("I", ward_minutes=ward_minutes)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(review.loc["VC-0347", "false_episodes"])
teach.check("Exercise I", answer)

#### Version 1.1 and the test of both versions

*Lecture page: [Improving the prototype: Version 1.1](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#improving-the-prototype-version-1-1)*

Version 1.1 adds the sample entropy to the four numbers, learns again on the training records and gets its own threshold on the validation records. Only then are the test records opened, once for each version, and the change log reports what improved and what it cost.

**From the lecture to the code.** The five numbers of version 1.1: `V1_1`. The model: `model_11`. The comparison on the test: `compare`. The change log: `changes`.

In [ ]:
V1_1 = V1_0 + ["sampen"]
model_11 = fit_model(V1_1, "1.1.0", training)
p_val_11 = af_probability(model_11, validation)
model_11["threshold"] = choose_threshold(p_val_11, validation["af"].to_numpy())

records_11 = by_record(validation, p_val_11, model_11["threshold"])
print("threshold", model_11["threshold"], "| validation AUC", round(roc_auc_score(validation["af"], p_val_11), 3))
print("false alarm episodes on the validation records: version 1.0", int(records_val["false_episodes"].sum()),
      "| version 1.1", int(records_11["false_episodes"].sum()))
print("false alarm minutes of record 08219:", int(((p_val >= model["threshold"]) & (validation["af"] == 0) & (validation["record"] == "08219")).sum()),
      "→", int(((p_val_11 >= model_11["threshold"]) & (validation["af"] == 0) & (validation["record"] == "08219")).sum()))

**Python note.** `V1_0 + ["sampen"]` makes a new list with five names; the list of version 1.0 stays as it was. `fit_model` and `choose_threshold` are the functions of «Stop 1 · Ready application», called with other arguments: The code of a version is the same, only its inputs change. The last line counts, with three masks joined by `&`, the minutes of record 08219 without AF that raise an alarm in each version.

In [ ]:
rows = []
for name, mdl in (("1.0", model), ("1.1", model_11)):
    p_te = af_probability(mdl, test)
    alarm = p_te >= mdl["threshold"]
    rec = by_record(test, p_te, mdl["threshold"])
    rows.append({"version": name, "AUC": round(roc_auc_score(test["af"], p_te), 3),
                 "sensitivity": round(alarm[test["af"].to_numpy() == 1].mean(), 3),
                 "specificity": round(1 - alarm[test["af"].to_numpy() == 0].mean(), 3),
                 "false alarm episodes": int(rec["false_episodes"].sum()),
                 "error of the AF burden (points)": round(float((rec["model_burden"] - rec["burden"]).abs().mean()), 1)})
compare = pd.DataFrame(rows)
print(compare.to_string(index=False))
digest_11 = save_model(model_11, "af_model_v1.1.0.json")
changes = {"version": "1.1.0", "change": "the sample entropy as a fifth number",
           "threshold": model_11["threshold"], "test: sensitivity": f"{compare.iloc[0, 2]} → {compare.iloc[1, 2]}",
           "test: false alarm episodes": f"{compare.iloc[0, 4]} → {compare.iloc[1, 4]}", "SHA-256": digest_11[:12]}
print(changes)

**Python note.** The loop judges both versions with the same code on the test records. `alarm[test["af"].to_numpy() == 1].mean()` is the share of the AF minutes with an alarm, the sensitivity, and one minus the share of the other minutes with an alarm is the specificity. `compare.iloc[0, 2]` reads the cell of the first row and the third column. `save_model` writes the file of version 1.1, which the web interface lists next to the file of version 1.0.

**Observe and reflect.** Version 1.1 gives fewer false alarms but misses more AF minutes on the test records. Which version would you choose for a ward that treats every AF episode with anticoagulation, and which for a ward that is drowning in alarms?

**Advanced (optional).** The noise of VC-0733 fools both versions. A check of the quality of the signal could mark minutes with missed or added beats, for example minutes in which an interval is shorter than 0.25 seconds or far longer than its neighbours, and leave them out of the alarm. Try such a rule on `ward_minutes` and count the false alarm episodes of VC-0733 again.

**Exercise J.** What is the sensitivity of version 1.1 on the test records? Keep it with three decimals.

In [ ]:
data = exercise_table("J", af_probability=af_probability, model_11=model_11, test=test)   # the table of the prompt of this exercise (Part F of the interactive lab)
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = float(compare.iloc[1]["sensitivity"])
teach.check("Exercise J", answer)

**Stop 3 is complete.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#ai-pair-programming-part-10-reviewing-a-prototype) and continue reading at «Working with an AI assistant, part 10: Reviewing a prototype». The weekly project is at the end of this Jupyter notebook, and the section «Weekly project» of the lecture page describes it.

## Weekly project

*Lecture page: [Weekly project](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#weekly-project) · [README of Week 10, Weekly project](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-10/README.md#weekly-project)*

**The project.** The prototype of the course tested as its users would test it, and improved: three tasks of a cardiologist on the telemetry ward done with version 1.0, the alarm episodes that the physician must review and their causes, an improved version with its threshold chosen on the validation records, and both versions compared once on the test records.

This part stands on its own: Its start cell loads the data of the project and prepares the tables of the steps. The project can be built with the prompts below, one for each step; each step works like a prompt of the stops, without a check cell or a ready code, because the results go into the report. The test of each prompt must hold before the next prompt is sent, and every prompt goes into the AI usage log.

**Start cell.** For the weekly project, the cell below repeats the setup and the data calls of this part, so that the code runs without errors. If you have run the earlier parts of this Jupyter notebook in this session, you may not need to run it; if in doubt, run it, since running it again does no harm.

In [ ]:
# Start cell: the weekly project
import pathlib
import sys
import urllib.request

if "vclinic" not in sys.modules:
    # the course package vclinic: the copy in the course folder, or else a copy downloaded once from GitHub
    folder = pathlib.Path.cwd()
    course = next((p for p in [folder, *folder.parents] if (p / "vclinic" / "__init__.py").exists()), None)
    if course is None:
        course = folder / "ai-cardiology-course"
        raw = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/vclinic/"
        (course / "vclinic").mkdir(parents=True, exist_ok=True)
        for file in urllib.request.urlopen(raw + "MANIFEST.txt").read().decode().split():
            urllib.request.urlretrieve(raw + file, course / "vclinic" / file)
    sys.path.insert(0, str(course))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import vclinic
from vclinic import teach

teach.start(week=10, language="en", packages=[])   # messages, figures, packages of the week and expected answers
RANDOM_STATE = 2026
from vclinic import afdb, holter

afdb.download()   # the MIT-BIH Atrial Fibrillation Database, downloaded once from PhysioNet

from vclinic import afdb, holter
import hashlib
import json
from sklearn.linear_model import LogisticRegression

_SETS = {"training": "training", "validation": "validation", "test": "test"}


def _course():
    """The minutes of the 25 records of the AFDB and of the four patients of the telemetry ward, with their RR intervals, their five numbers and the probabilities of version 1.0 (computed once, about twenty seconds)."""
    if "_course_cache" in globals():
        return _course_cache
    SPLIT = {record: part for part, records in afdb.SPLITS.items() for record in records}
    episodes = {record: afdb.rhythm(record) for record in afdb.RECORDS}
    rhythms = pd.DataFrame([{"record": record, "rhythm": name, "hours": (end - start) / 3600}
                            for record in afdb.RECORDS for start, end, name in episodes[record]])
    hours = rhythms.pivot_table(index="record", columns="rhythm", values="hours", aggfunc="sum", fill_value=0.0)
    hours["set"] = hours.index.map(SPLIT)

    WINDOW_S = 60


    def minutes_of(beat_s, episodes=None, record=""):
        """One-minute windows of a recording: the RR intervals that end in each minute and, when the rhythm is known, the share of the minute in AF."""
        rr = np.diff(beat_s)
        rr_end = beat_s[1:]
        rows = []
        for start in np.arange(beat_s[0], beat_s[-1] - WINDOW_S, WINDOW_S):
            x = rr[(rr_end >= start) & (rr_end < start + WINDOW_S)]
            if len(x) < 20 or x.max() > 3.0:
                continue
            row = {"record": record, "start_s": float(start), "rr": x}
            if episodes is not None:
                af_s = sum(max(0.0, min(e, start + WINDOW_S) - max(s, start)) for s, e, name in episodes if name == "(AFIB")
                row["af_share"] = af_s / WINDOW_S
            rows.append(row)
        return rows

    minutes = pd.DataFrame([row for record in afdb.RECORDS for row in minutes_of(afdb.beats(record), episodes[record], record)])
    minutes["af"] = (minutes["af_share"] > 0.5).astype(int)
    minutes["set"] = minutes["record"].map(SPLIT)


    def sample_entropy(rr, m=1, r=0.03):
        """Sample entropy: minus the logarithm of the probability that two stretches of m + 1 intervals are alike, within r seconds, when their first m intervals are alike."""
        n = len(rr)

        def pairs(length):
            t = np.lib.stride_tricks.sliding_window_view(rr, length)[: n - m]
            alike = np.abs(t[:, None, :] - t[None, :, :]).max(axis=2) <= r
            return (alike.sum() - len(t)) / 2

        b, a = pairs(m), pairs(m + 1)
        return float(np.log(b / max(a, 1))) if b > 0 else float(np.log(n))


    def features(rr):
        """The five numbers of a minute: heart rate, coefficient of variation, RMSSD divided by the mean RR, pNN50 and sample entropy."""
        d = np.diff(rr)
        return {"hr": 60 / rr.mean(), "cv": rr.std(ddof=1) / rr.mean(), "rmssd": np.sqrt(np.mean(d ** 2)) / rr.mean(),
                "pnn50": np.mean(np.abs(d) > 0.05), "sampen": sample_entropy(rr)}

    minutes = minutes.join(pd.DataFrame([features(x) for x in minutes["rr"]], index=minutes.index))

    V1_0 = ["hr", "cv", "rmssd", "pnn50"]
    LOGGED = ["cv", "rmssd"]


    def design(table, names):
        """The features of the model: the logarithm of the CV and of the RMSSD, the others as they are."""
        X = table[names].astype(float).copy()
        for name in names:
            if name in LOGGED:
                X[name] = np.log(X[name])
        return X


    def fit_model(names, version, training):
        """A logistic regression on standardised features, kept as a plain dictionary that JSON can store."""
        X = design(training, names)
        mean, sd = X.mean(), X.std()
        lr = LogisticRegression(max_iter=3000).fit((X - mean) / sd, training["af"])
        return {"name": "AF from RR intervals", "version": version, "features": names,
                "mean": {k: round(float(v), 6) for k, v in mean.items()}, "sd": {k: round(float(v), 6) for k, v in sd.items()},
                "coef": {k: round(float(v), 6) for k, v in zip(names, lr.coef_[0])}, "intercept": round(float(lr.intercept_[0]), 6)}


    def af_probability(model, table):
        """The probability of AF for every minute of a table."""
        names = model["features"]
        z = (design(table, names) - pd.Series(model["mean"])[names]) / pd.Series(model["sd"])[names]
        w = np.array([model["coef"][name] for name in names])
        return 1 / (1 + np.exp(-(model["intercept"] + z[names].to_numpy() @ w)))


    def choose_threshold(p, y, sensitivity=0.95):
        """The highest threshold, in steps of 0.01, that still finds the given share of the AF minutes."""
        grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
        return float(max(t for t in grid if (p[y == 1] >= t).mean() >= sensitivity))


    def alarm_episodes(alarm):
        """Runs of consecutive alarm minutes, as (first, last) positions."""
        runs, first = [], None
        for k, a in enumerate(alarm):
            if a and first is None:
                first = k
            if not a and first is not None:
                runs.append((first, k - 1))
                first = None
        if first is not None:
            runs.append((first, len(alarm) - 1))
        return runs


    def by_record(table, p, threshold):
        """For every record: the AF burden of the annotation and of the model, the alarm episodes and the false ones among them."""
        rows = []
        for record, part in table.assign(p=p).groupby("record", sort=True):
            alarm = (part["p"] >= threshold).to_numpy()
            runs = alarm_episodes(alarm)
            false = sum(1 for a, b in runs if part["af"].to_numpy()[a:b + 1].sum() == 0)
            rows.append({"record": record, "minutes": len(part), "burden": 100 * part["af_share"].mean(),
                         "model_burden": 100 * alarm.mean(), "episodes": len(runs), "false_episodes": false})
        return pd.DataFrame(rows).set_index("record")

    training, validation, test = (minutes[minutes["set"] == s] for s in ("training", "validation", "test"))
    model = fit_model(V1_0, "1.0.0", training)
    p_val = af_probability(model, validation)
    model["threshold"] = choose_threshold(p_val, validation["af"].to_numpy())


    def contributions(model, row):
        """What each feature adds to the log-odds of AF in one minute: its coefficient times its z-score."""
        x = design(pd.DataFrame([row]), model["features"]).iloc[0]
        z = (x - pd.Series(model["mean"])) / pd.Series(model["sd"])
        return {name: model["coef"][name] * z[name] for name in model["features"]}


    def predict(request, model, digest):
        """The prediction service: a request with the beat times of a recording in, the result out, both as dictionaries that JSON can carry."""
        table = pd.DataFrame(minutes_of(np.asarray(request["beat_times_s"], dtype=float)))
        table = table.join(pd.DataFrame([features(x) for x in table["rr"]], index=table.index))
        p = af_probability(model, table)
        alarm = p >= model["threshold"]
        runs = alarm_episodes(alarm)
        return {"model": model["name"], "version": model["version"], "sha256": digest, "threshold": model["threshold"],
                "minutes": int(len(table)), "af_minutes": int(alarm.sum()), "burden_percent": round(100 * float(alarm.mean()), 1),
                "episodes": [{"start_s": table["start_s"].iloc[a], "minutes": b - a + 1} for a, b in runs]}

    ward = holter.ward()
    ward_minutes = pd.DataFrame([row for pid, rec in ward.items()
                                 for row in minutes_of(rec["detected_s"], [(a, b, "(AFIB") for a, b in rec["episodes"]], pid)])
    ward_minutes["af"] = (ward_minutes["af_share"] > 0.5).astype(int)
    ward_minutes = ward_minutes.join(pd.DataFrame([features(x) for x in ward_minutes["rr"]], index=ward_minutes.index))
    minutes["p10"] = af_probability(model, minutes)
    ward_minutes["p10"] = af_probability(model, ward_minutes)
    digest = hashlib.sha256(json.dumps(model, indent=2, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
    globals()["_course_cache"] = dict(minutes=minutes, model=model, digest=digest, ward=ward, ward_minutes=ward_minutes,
                                      contributions=contributions, predict=predict, design=design, by_record=by_record)
    return _course_cache


def _minutes_table(columns, sets=("training", "validation", "test")):
    """A table of the minutes of the AFDB with the chosen columns."""
    m = _course()["minutes"]
    m = m[m["set"].isin(sets)]
    cols = {"record": m["record"], "set": m["set"].map(_SETS), "start (s)": m["start_s"].round(2),
            "AF": m["af"], "AF share": m["af_share"].round(3), "RR intervals (s)": [list(np.round(x, 3)) for x in m["rr"]],
            "heart rate": m["hr"].round(2), "CV": m["cv"].round(4), "RMSSD/RR": m["rmssd"].round(4), "pNN50": m["pnn50"].round(4),
            "probability": m["p10"].round(4)}
    return pd.DataFrame({k: v for k, v in cols.items() if k in columns}).reset_index(drop=True)



def _project_minutes():
    """The validation minutes of the AFDB and the minutes of the telemetry ward, with their RR intervals, the four numbers and the probability of version 1.0."""
    c = _course()
    v = c["minutes"][c["minutes"]["set"] == "validation"]
    w = c["ward_minutes"]
    rows = []
    for src, t in (("AFDB", v), ("ward", w)):
        rows.append(pd.DataFrame({"source": src, "recording": t["record"].to_numpy(), "start (h)": (t["start_s"] / 3600).round(4).to_numpy(),
                                  "AF": t["af"].to_numpy(), "probability, version 1.0": t["p10"].round(4).to_numpy()}))
    return pd.concat(rows, ignore_index=True)

### Project step 1: The alarm episodes of the user test

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist runs a user test of an AF prototype on five validation records and on four patients of a telemetry ward, and needs the list of alarm episodes to review, one row per episode.  
> Data: The notebook holds a table named "data" with the minutes of the user test: the 5 validation records of the MIT-BIH Atrial Fibrillation Database and the 24-hour recordings of the four patients of the telemetry ward of the virtual clinic, a synthetic data set. Each row is a minute: "source" gives "AFDB" or "ward", "recording" the record or the patient, and "start (h)" the start of the minute in hours from the start of the recording. "AF" is 1 when more than half of the minute lies in atrial fibrillation (AF) in the annotation or the truth of the recording, and "probability, version 1.0" is the probability of AF from version 1.0 of a model of the course, which raises an alarm at 0.37 or more.  
> Task: Within each recording, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm. Make one row per alarm episode with the source, the recording, the start in hours, the length in minutes, the highest probability and whether any minute of the episode has AF.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the first rows of the table and the number of episodes of each recording, and keep the total number of alarm episodes under the name "answer".  
> Test: The lengths of the episodes must add up to the minutes with an alarm.  

In [ ]:
# Data of project step 1: the table "data" that the prompt describes
data = _project_minutes()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 2: The workload of the physician

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist wants to know how many alarm episodes a physician would review per 24 hours with version 1.0, how many of them are false, and in which recordings the false ones gather.  
> Data: The notebook holds a table named "data" with the minutes of the user test: the 5 validation records of the MIT-BIH Atrial Fibrillation Database and the 24-hour recordings of the four patients of the telemetry ward of the virtual clinic, a synthetic data set. Each row is a minute: "source" gives "AFDB" or "ward", "recording" the record or the patient, and "start (h)" the start of the minute in hours from the start of the recording. "AF" is 1 when more than half of the minute lies in atrial fibrillation (AF) in the annotation or the truth of the recording, and "probability, version 1.0" is the probability of AF from version 1.0 of a model of the course, which raises an alarm at 0.37 or more.  
> Task: Within each record, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm, and a false alarm episode is an episode in which no minute has AF. For each recording, count the minutes, the alarm episodes and the false alarm episodes, and give the alarm episodes and the false ones per 24 hours of recording. Then give the same for the AFDB and for the ward as a whole.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show a table with one row for each recording and the two totals, and keep the share of false alarm episodes over all recordings, with three decimals, under the name "answer".  
> Test: The false alarm episodes of a recording cannot exceed its alarm episodes.  

In [ ]:
# Data of project step 2: the table "data" that the prompt describes
data = _project_minutes()
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 3: An improved version on the validation records

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The user test showed that premature beats raise most false alarms. The cardiologist wants an improved version, fitted on the training records and judged on the validation records only.  
> Data: The notebook holds a table named "data" with the minutes of the MIT-BIH Atrial Fibrillation Database (AFDB), 25 Holter recordings of about ten hours of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. The test records are not in it. Each row is a minute: "record" gives the recording, "set" its set ("training", 15 records; "validation", 5 records), and "start (s)" the start of the minute in seconds from the start of the recording. "AF" is 1 when more than half of the minute lies in AF as marked by the experts, and 0 otherwise. Each cell of "RR intervals (s)" holds the RR intervals of the minute in seconds, in their order, as a list of 20 or more numbers. The four numbers of the minute are "heart rate", "CV", "RMSSD/RR" and "pNN50".  
> Task: For every minute, also compute the sample entropy of its RR intervals with stretches of one interval and a tolerance of 0.03 seconds: Of the first n − 1 intervals of the minute, count the pairs that differ by at most 0.03 seconds (B), and count how many of these pairs are still alike when each interval is followed by the next one (A). The sample entropy is the natural logarithm of B divided by A; count A as 1 when it is 0, and give the minute the natural logarithm of n when B is 0. Fit a logistic regression for AF on the training minutes with at most 3,000 iterations, on five inputs: the heart rate, the natural logarithm of the CV, the natural logarithm of the RMSSD/RR, the pNN50 and the sample entropy, each turned into a z-score with the mean and the standard deviation of the training minutes. Choose the threshold on the validation minutes only: Among 0.05, 0.06, 0.07 and so on up to 0.95, take the highest value at which at least 95 percent of the validation minutes with AF have a probability at or above it. Give the validation AUC, the sensitivity and the specificity at the threshold, and the false alarm episodes of every validation record.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the results, print the coefficients and the threshold, and keep the threshold, with two decimals, under the name "answer".  
> Test: No test minute may be used, and the sensitivity on the validation records must be at least 0.95.  

In [ ]:
# Data of project step 3: the table "data" that the prompt describes
data = _minutes_table(["record", "set", "start (s)", "AF", "RR intervals (s)",
                       "heart rate", "CV", "RMSSD/RR", "pNN50"], sets=("training", "validation"))
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

### Project step 4: Both versions on the test records and the change log

**The prompt.** Copy the text below and send it to your AI assistant. The data cell right below it prepares the table "data" that the prompt describes.

> Role: You are helping a cardiologist who does not write code. The code will be pasted into a Jupyter notebook and run there.  
> Context: The cardiologist opens the test records once for both versions and writes the change log of the new version, with its gains and its costs.  
> Data: The notebook holds a table named "data" with the minutes of the 5 test records of the MIT-BIH Atrial Fibrillation Database, Holter recordings of people with mostly paroxysmal atrial fibrillation (AF), cut into windows of one minute. Each row is a minute: "record" gives the recording and "start (s)" the start of the minute in seconds; "AF" is 1 when more than half of the minute lies in AF as marked by the experts, and "AF share" is the share of the minute in AF. Each cell of "RR intervals (s)" holds the RR intervals of the minute in seconds, and the four numbers of the minute are "heart rate", "CV", "RMSSD/RR" and "pNN50".  
> Task: Version 1.0 uses four inputs: the heart rate, the natural logarithm of the CV, the natural logarithm of the RMSSD/RR and the pNN50; version 1.1 adds the sample entropy of the RR intervals with stretches of one interval and a tolerance of 0.03 seconds. Use the two versions of step 3 and of Stop 1 exactly as they were fitted on the training minutes, with their thresholds from the validation minutes. Within each record, sort the minutes by their start. An alarm episode is a run of consecutive minutes with an alarm, and a false alarm episode is an episode in which no minute has AF. For each version, compute on the test minutes the AUC, the sensitivity, the specificity, the false alarm episodes and the mean absolute error of the AF burden per record, in percentage points. Then write a change log of the new version with its version number, the change, the reason, the threshold, the effects on the validation and on the test records, and what did not improve.  
> Constraints: Use only this table. The code must not read files, download anything from the internet or install programs.  
> Output: Show the table of the two versions and print the change log, and keep the sensitivity of the new version on the test records, with three decimals, under the name "answer".  
> Test: The test records must be used once for each version, and no threshold may be changed after looking at them.  

In [ ]:
# Data of project step 4: the table "data" that the prompt describes
data = _minutes_table(["record", "start (s)", "AF", "AF share", "RR intervals (s)",
                       "heart rate", "CV", "RMSSD/RR", "pNN50"], sets=("test",))
answer = None   # the answer of an earlier prompt is cleared
data

In [ ]:
# Paste the code of the AI assistant here, in place of this line, and run the cell.

**End of the weekly project.** [Go back to the lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-10/lecture.html#weekly-project): The section «Weekly project» describes the report and its submission.

## Application challenges by field

| Field | Challenge |
|---|---|
| Telemetry and coronary care | Write the workflow of your ward for an alarm of AF: who sees it, within which time, on which screen, and what the decision log must hold. Count the alarm episodes of version 1.0 per patient and day in «Stop 3 · Ready application» and decide whether your nurses could review them. |
| Electrophysiology | After an ablation, the AF burden of a Holter recording is an outcome. Compute the burden of every test record with both versions and judge whether the error of version 1.1 is small enough to compare a patient before and after the ablation. |
| Stroke prevention | A missed episode of AF may mean a missed anticoagulation. Find the threshold of version 1.0 on the validation records that keeps a sensitivity of 0.99, and report what it does to the false alarm episodes. |
| Heart failure clinic | Patients with heart failure often have premature beats. Use «Stop 3 · Ready application» to list which numbers of a minute move most between the hours with and without premature beats of VC-0347, and propose a sentence for the model card on patients with frequent ectopy. |
| Wearables and remote monitoring | A watch gives the times of the pulse, not of the R peaks, and it misses more beats. Remove one beat in twenty at random from the beats of VC-0128, run `predict` again and compare the AF burden with that of the full recording. |
| Quality and patient safety | Use the decision log of «Stop 3 · Ready application» as a monitor: Compute the share of rejected alarm episodes per patient, and write the rule that would make the clinic review the version of the model when this share rises. |
| Research and ethics | Write the half page that an ethics committee would need for a study of the prototype on your ward: the intended use, the data and where they stay, the information of the patients, the decision of the physician and the plan for false alarms. |

## After the notebook

The prototype of this week reads the beats of a monitor, explains its alarms, writes a report that other systems can read and keeps the version of its model in every output. The weekly project tests it on its users and improves it. Week 11 turns to the final project: a question of your own clinic, from the data to a model, its explanation and the design of its clinical evaluation.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>